# SKU-based demand forecasting

Selected original implementation, cleaned for public review. No commercial data or historical outputs are included. Run only with authorized local inputs. Configuration values are examples, not validated business settings.

In [ ]:
from pathlib import Path
import os

# Run from the repository root or notebooks directory.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
for folder in ['Input', 'Clustering', 'PreEstimation', 'Modeling', 'SubCatStrategy/Input', 'SubCatStrategy/Data', 'SubCatStrategy/SubCatForecast']:
    Path("workspace", folder).mkdir(parents=True, exist_ok=True)

## Calendar-aware input preparation

In [ ]:
def build_input_with_hijri_main_and_secondary(input_path, output_path):
    """
    Creates an Excel with:
      - Date_Gregorian        : first day of each Gregorian month (datetime)
      - Hijri_Month_Main      : Hijri month that covers most days of that Gregorian month (1–12)
      - Hijri_Month_Secondary : the other Hijri month in that Gregorian month (if exists, else None)
      - Material Code
      - Sku Name
      - Vol (Kg)

    Hijri engine:
      1) hijri_converter  (if available)
      2) hijridate        (if available)
      3) manual fallback  (approximate if neither exists)
    """

    import pandas as pd
    import calendar

    # -------------------------------
    # 1) Hijri Conversion Engine – سه حالت
    # -------------------------------
    HIJRI_ENGINE = None
    Hijri = None
    Gregorian = None

    # حالت 1: hijri_converter
    try:
        from hijri_converter import Hijri as HC_Hijri, Gregorian as HC_Gregorian
        Hijri = HC_Hijri
        Gregorian = HC_Gregorian
        HIJRI_ENGINE = "hijri_converter"
        print("[INFO] Hijri engine: hijri_converter")
    except Exception:
        pass

    # حالت 2: hijridate
    if HIJRI_ENGINE is None:
        try:
            from hijridate import Hijri as HD_Hijri, Gregorian as HD_Gregorian
            Hijri = HD_Hijri
            Gregorian = HD_Gregorian
            HIJRI_ENGINE = "hijridate"
            print("[INFO] Hijri engine: hijridate")
        except Exception:
            pass

    # حالت 3: manual fallback
    if HIJRI_ENGINE is None:
        HIJRI_ENGINE = "manual"
        print("[WARNING] Using manual Hijri conversion (approximate).")

    # -------------------------------
    # 2) Manual fallback algorithm
    # -------------------------------
    def gregorian_to_hijri_manual(gdate):
        """
        Approximate fallback for Gregorian → Hijri.
        Accuracy is lower than proper libraries but avoids empty columns.
        """
        if pd.isna(gdate):
            return (None, None, None)

        gy = gdate.year
        doy = gdate.timetuple().tm_yday

        # approximate Hijri year
        hy = int(round((gy - 622) * 33.0 / 32.0))

        greg_year_days = 365.242
        hijri_year_days = 354.367

        frac = doy / greg_year_days
        day_in_hijri_year = int(round(frac * hijri_year_days))
        if day_in_hijri_year <= 0:
            day_in_hijri_year = 1

        # tabular month lengths 30/29
        month_lengths = [30, 29] * 6  # 12 Hijri months
        hm = 1
        hd = day_in_hijri_year
        for ml in month_lengths:
            if hd > ml:
                hd -= ml
                hm += 1
            else:
                break

        if hm > 12:
            hm = 12

        return (hy, hm, hd)

    # -------------------------------
    # 3) Safe conversion wrapper
    # -------------------------------
    def gregorian_to_hijri_safe(gdate):
        """
        Uses hijri_converter / hijridate if available,
        otherwise uses the manual fallback.
        Returns: (h_year, h_month, h_day)
        """
        if pd.isna(gdate):
            return (None, None, None)

        if HIJRI_ENGINE in ("hijri_converter", "hijridate") and Gregorian is not None:
            try:
                g = Gregorian(gdate.year, gdate.month, gdate.day)
                h = g.to_hijri()
                return (h.year, h.month, h.day)
            except Exception as e:
                print(f"[WARNING] Hijri conversion failed for {gdate} via {HIJRI_ENGINE}: {e}")

        # Fallback
        return gregorian_to_hijri_manual(gdate)

    # -------------------------------
    # 4) Load input data
    # -------------------------------
    df = pd.read_excel(input_path)

    # Expect columns:
    # 'year', 'Month', 'Material Code', 'Sku Name', 'Vol (Kg)'
    required_cols = ["year", "Month", "Material Code", "Sku Name", "Vol (Kg)"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in input: {missing}")

    # -------------------------------
    # 5) Build Gregorian first day of month
    # -------------------------------
    df["Date_Gregorian"] = pd.to_datetime(
        df["year"].astype(str) + "-" + df["Month"].astype(str) + "-01",
        errors="raise"
    )

    # -------------------------------
    # 6) For each Gregorian month, compute dominant & secondary Hijri months
    # -------------------------------
    month_map = {}  # key: Date_Gregorian (month start), value: (main_hm, secondary_hm)

    unique_month_starts = df["Date_Gregorian"].dropna().unique()
    for ms in unique_month_starts:
        ts = pd.Timestamp(ms)
        year = ts.year
        month = ts.month
        days_in_month = calendar.monthrange(year, month)[1]

        counts = {}  # hijri_month -> number of days in this greg month
        for day in range(1, days_in_month + 1):
            gdate = pd.Timestamp(year, month, day)
            _, hm, _ = gregorian_to_hijri_safe(gdate)
            if hm is None:
                continue
            counts[hm] = counts.get(hm, 0) + 1

        if not counts:
            month_map[ms] = (None, None)
        else:
            # sort by coverage (descending)
            sorted_months = sorted(counts.items(), key=lambda x: x[1], reverse=True)
            main_hm = sorted_months[0][0]
            secondary_hm = sorted_months[1][0] if len(sorted_months) > 1 else None
            month_map[ms] = (main_hm, secondary_hm)

    # Map back to all rows
    df["Hijri_Month_Main"] = df["Date_Gregorian"].map(lambda d: month_map.get(d, (None, None))[0])
    df["Hijri_Month_Secondary"] = df["Date_Gregorian"].map(lambda d: month_map.get(d, (None, None))[1])

    # -------------------------------
    # 7) Final output columns
    # -------------------------------
    out_cols = [
        "Date_Gregorian",
        "Hijri_Month_Main",
        "Hijri_Month_Secondary",
        "Material Code",
        "Sku Name",
        "Vol (Kg)"
    ]
    df_out = df[out_cols].copy()

    # -------------------------------
    # 8) Save to Excel
    # -------------------------------
    df_out.to_excel(output_path, index=False)
    print(f"[INFO] Output saved to: {output_path}")

## Time-series clustering and demand segmentation

In [ ]:
def cluster_time_series_data(
    raw_input_path,
    hijri_output_path,
    k_list=(3, 4, 5, 6, 7, 8, 9, 10),
    models=("kmeans", "agglo", "gmm"),
    manual_cluster_path=None,
    final_model_name=None,   # chosen model for Cluster_Final
    final_k=None,            # chosen k for Cluster_Final
):
    """
    Full clustering + segmentation + meta-clustering pipeline.

    1) Calls build_input_with_hijri_main_and_secondary to generate the time-series input
       and save it to hijri_output_path (e.g., inputv2).
    2) Loads that file (with Date_Gregorian, Material Code, Vol (Kg), ...).
    3) Builds a SKU × month matrix of sales volume.
    4) Computes segmentations:
         - ABC segmentation (A/B/C) based on total volume:
             * A: top ~80% of cumulative volume
             * B: next ~15% (up to 95%)
             * C: remaining ~5% of volume
         - XYZ segmentation (X/Y/Z) based on CV of monthly demand:
             * X: CV < 0.20  (CV < 20%)
             * Y: 0.20 ≤ CV < 0.50  (20%–50%)
             * Z: CV ≥ 0.50  (≥50%)
         - ABCXYZ combined code (AX, BY, CZ, …)
         - Demand Pattern (Smooth / Intermittent / Erratic / Lumpy)
           using ADI & CV² thresholds (1.32, 0.49)
    5) Runs FIRST-LEVEL clustering on SKU×month features:
         - models: kmeans, agglo, gmm
         - k in k_list (subject to n_samples)
         - PLUS: hierarchical time-series clustering ("hier_ts") using
           Ward linkage on standardized monthly volume.
         - For each, compute silhouette.
    6) Treat segmentations also as clustering models:
         - ABC, XYZ, ABCXYZ, DemandPattern
         - Compute silhouette for each on SKU×month features.
    7) Runs META-CLUSTERING:
         - Build per-SKU feature matrix from ONLY:
             * ABCXYZ_Class (encoded)
             * Demand_Pattern (encoded)
             * first-level KMeans k=5 cluster (encoded)
         - Standardize features; run KMeans with k=3..10
         - Compute silhouette on this meta-feature space.
    8) If manual_cluster_path is provided:
         - Load ClusterManual.xlsx (first two columns: Material Code, cluster)
         - Map to pivot SKUs
         - Add a clustering model "Manual" with k = #unique manual clusters
           and compute silhouette.
    9) Build a unified results dictionary and summary_df:
         - results[(model_name, k)] -> pd.Series(cluster labels, index=SKU)
         - summary_df rows: model, k, sample_size, metric="silhouette", metric_value
    10) Write two Excel files:
         10.1) workspace/Clustering/ClusterV1_detail.xlsx
               → row-level, same as inputv2 + segmentations + ALL cluster columns
         10.2) workspace/Clustering/ClusterV1.xlsx
               → SKU-level summary:
                  - removed: Date_Gregorian, Hijri_Month_Main, Hijri_Month_Secondary, Vol (Kg)
                  - one row per Material Code
                  - includes ABC/XYZ/ABCXYZ, ADI, CV, CV2, Demand_Pattern,
                    and ALL cluster columns (first-level + ABC/XYZ/Demand + meta + Manual).
    11) If final_model_name & final_k are provided and valid:
         - Adds a 'Cluster_Final' column to df_aug and df_sku
         - Writes:
             * workspace/Clustering/Cluster_Final.xlsx
               (Material Code, Sku Name, Cluster_Final)
             * workspace/Clustering/Cluster_Final_detail.xlsx
               (Date_Gregorian, Hijri_Month_Main, Hijri_Month_Secondary,
                Material Code, Sku Name, Cluster_Final, Vol (Kg))
             * workspace/Clustering/Cluster_Final_Aggregated.xlsx
               (Date_Gregorian, Hijri_Month_Main, Hijri_Month_Secondary,
                Cluster_Final, Vol (Kg) [aggregated per date+cluster])
    12) Returns:
         - df_ts       : augmented time-series dataframe (row-level)
         - pivot_table : SKU × month_period matrix
         - results     : dict[(model, k)] -> pd.Series(cluster labels, index=SKU)
         - summary_df  : DataFrame with metrics per (model, k)
    """

    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans, AgglomerativeClustering
    from sklearn.mixture import GaussianMixture
    from sklearn.metrics import silhouette_score

    # -----------------------------------
    # Helper: safe silhouette computation
    # -----------------------------------
    def compute_silhouette(X_data, labels_array):
        metric_value = np.nan
        labels_array = np.asarray(labels_array)
        mask = ~pd.isna(labels_array)
        if mask.sum() <= 1:
            return metric_value
        lbl = labels_array[mask]
        if len(np.unique(lbl)) <= 1:
            return metric_value
        try:
            metric_value = float(silhouette_score(X_data[mask], lbl))
        except Exception as e:
            print(f"[WARNING] silhouette_score failed: {e}")
        return metric_value

    # -----------------------------------
    # 1) Build input via your hijri engine (inputv2-style)
    # -----------------------------------
    build_input_with_hijri_main_and_secondary(
        input_path=raw_input_path,
        output_path=hijri_output_path
    )

    # -----------------------------------
    # 2) Load the transformed dataset (inputv2)
    # -----------------------------------
    df = pd.read_excel(hijri_output_path)

    DATE_COL = "Date_Gregorian"
    SKU_COL = "Material Code"
    VOL_COL = "Vol (Kg)"

    if DATE_COL not in df.columns or SKU_COL not in df.columns or VOL_COL not in df.columns:
        raise ValueError(
            f"Expected columns '{DATE_COL}', '{SKU_COL}', '{VOL_COL}' not all found in the file."
        )

    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    # -----------------------------------
    # 3) Add time helper columns
    # -----------------------------------
    df["year"] = df[DATE_COL].dt.year
    df["month"] = df[DATE_COL].dt.month
    df["month_period"] = df[DATE_COL].dt.to_period("M")

    # -----------------------------------
    # 4) Build SKU × month matrix
    # -----------------------------------
    pivot = (
        df.pivot_table(
            index=SKU_COL,
            columns="month_period",
            values=VOL_COL,
            aggfunc="sum"
        )
        .fillna(0.0)
    )

    pivot = pivot.sort_index(axis=1)

    if pivot.empty:
        raise ValueError("Pivot table is empty – check input data.")

    # ============================================================
    # 5) ABC–XYZ SEGMENTATION (per SKU)
    # ============================================================
    total_vol = pivot.sum(axis=1)
    total_sorted = total_vol.sort_values(ascending=False)
    cum_share = total_sorted.cumsum() / total_sorted.sum()

    abc_class_sorted = pd.Series(index=total_sorted.index, dtype="object")
    abc_class_sorted[cum_share <= 0.80] = "A"
    abc_class_sorted[(cum_share > 0.80) & (cum_share <= 0.95)] = "B"
    abc_class_sorted[cum_share > 0.95] = "C"
    abc_class = abc_class_sorted.reindex(pivot.index)

    mean_monthly = pivot.mean(axis=1)
    std_monthly = pivot.std(axis=1, ddof=0)

    cv = std_monthly / (mean_monthly.replace(0, np.nan))
    cv = cv.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    xyz_class = pd.Series(index=pivot.index, dtype="object")
    xyz_class[cv < 0.20] = "X"
    xyz_class[(cv >= 0.20) & (cv < 0.50)] = "Y"
    xyz_class[cv >= 0.50] = "Z"

    abcxyz_class = abc_class.astype(str) + xyz_class.astype(str)

    # ============================================================
    # 6) DEMAND PATTERN (ADI & CV² → Smooth / Intermittent / Erratic / Lumpy)
    # ============================================================
    ADI = pd.Series(index=pivot.index, dtype=float)
    CV2 = pd.Series(index=pivot.index, dtype=float)

    n_periods = pivot.shape[1]

    for sku, row in pivot.iterrows():
        vals = row.values
        T = n_periods

        nonzero_vals = vals[vals > 0]
        n_z = len(nonzero_vals)

        if n_z == 0:
            adi_val = T + 1.0
            cv2_val = 0.0
        else:
            adi_val = T / float(n_z)
            mean_nz = nonzero_vals.mean()
            if mean_nz == 0:
                cv2_val = 0.0
            else:
                std_nz = nonzero_vals.std(ddof=0)
                cv_val_local = std_nz / mean_nz
                cv2_val = float(cv_val_local ** 2)

        ADI.loc[sku] = adi_val
        CV2.loc[sku] = cv2_val

    pattern = pd.Series(index=pivot.index, dtype="object")
    smooth_mask = (ADI < 1.32) & (CV2 < 0.49)
    intermittent_mask = (ADI >= 1.32) & (CV2 < 0.49)
    erratic_mask = (ADI < 1.32) & (CV2 >= 0.49)
    lumpy_mask = (ADI >= 1.32) & (CV2 >= 0.49)

    pattern[smooth_mask] = "Smooth"
    pattern[intermittent_mask] = "Intermittent"
    pattern[erratic_mask] = "Erratic"
    pattern[lumpy_mask] = "Lumpy"

    # ============================================================
    # 7) Attach segmentations to df (row-level by Material Code)
    # ============================================================
    df["ABC_Class"] = df[SKU_COL].map(abc_class)
    df["XYZ_Class"] = df[SKU_COL].map(xyz_class)
    df["ABCXYZ_Class"] = df[SKU_COL].map(abcxyz_class)
    df["ADI"] = df[SKU_COL].map(ADI)
    df["CV"] = df[SKU_COL].map(cv)
    df["CV2"] = df[SKU_COL].map(CV2)
    df["Demand_Pattern"] = df[SKU_COL].map(pattern)

    print("\n[INFO] ABC–XYZ segmentation (ABCXYZ counts):")
    print(abcxyz_class.value_counts().sort_index())

    print("\n[INFO] Demand pattern segmentation (counts):")
    print(pattern.value_counts().sort_index())

    # -----------------------------------
    # 8) FIRST-LEVEL clustering on SKU×month features
    # -----------------------------------
    scaler = StandardScaler()
    X = scaler.fit_transform(pivot.values)

    results = {}
    rows = []

    n_samples = X.shape[0]

    # 8a) KMeans / Agglo / GMM
    for model_name in models:
        for k in k_list:
            if k < 2 or k > n_samples:
                continue

            labels = None
            extra = {}
            metric = "silhouette"
            metric_value = np.nan

            if model_name == "kmeans":
                model = KMeans(
                    n_clusters=k,
                    random_state=42,
                    n_init="auto"
                )
                labels = model.fit_predict(X)
                extra["inertia"] = float(model.inertia_)

            elif model_name == "agglo":
                model = AgglomerativeClustering(
                    n_clusters=k,
                    linkage="ward"
                )
                labels = model.fit_predict(X)

            elif model_name == "gmm":
                model = GaussianMixture(
                    n_components=k,
                    random_state=42
                )
                labels = model.fit_predict(X)
                try:
                    extra["aic"] = float(model.aic(X))
                    extra["bic"] = float(model.bic(X))
                except Exception:
                    pass

            else:
                continue

            if labels is None:
                continue

            metric_value = compute_silhouette(X, labels)

            labels_series = pd.Series(
                labels,
                index=pivot.index,
                name="cluster"
            )
            results[(model_name, k)] = labels_series

            row = {
                "model": model_name,
                "k": int(k),
                "sample_size": n_samples,
                "metric": metric,
                "metric_value": metric_value,
            }
            row.update(extra)
            rows.append(row)

    # 8b) Hierarchical time-series clustering (Ward / Euclidean)
    try:
        from scipy.cluster.hierarchy import linkage, fcluster
        print("\n[INFO] Running hierarchical time-series clustering (Ward/Euclidean)...")
        Z = linkage(X, method="ward", metric="euclidean")

        for k in k_list:
            if k < 2 or k > n_samples:
                continue

            labels = fcluster(Z, t=k, criterion="maxclust") - 1
            metric_value = compute_silhouette(X, labels)

            labels_series = pd.Series(
                labels,
                index=pivot.index,
                name="cluster"
            )
            results[("hier_ts", k)] = labels_series

            rows.append({
                "model": "hier_ts",
                "k": int(k),
                "sample_size": n_samples,
                "metric": "silhouette",
                "metric_value": metric_value,
            })
    except Exception as e:
        print(f"[WARNING] Hierarchical time-series clustering skipped: {e}")

    # -----------------------------------
    # 9) Add segmentation models as clusterings
    # -----------------------------------
    seg_models = [
        ("ABC", abc_class),
        ("XYZ", xyz_class),
        ("ABCXYZ", abcxyz_class),
        ("DemandPattern", pattern),
    ]

    for model_name, label_series in seg_models:
        labels_array = label_series.reindex(pivot.index).values
        unique_labels = pd.unique(label_series.dropna())
        k_seg = len(unique_labels)
        if k_seg < 2:
            continue

        metric_value = compute_silhouette(X, labels_array)

        results[(model_name, k_seg)] = label_series

        rows.append({
            "model": model_name,
            "k": int(k_seg),
            "sample_size": n_samples,
            "metric": "silhouette",
            "metric_value": metric_value,
        })

    # -----------------------------------
    # 10) Ensure we have FIRST-LEVEL KMeans k=5 (needed for meta features)
    # -----------------------------------
    if ("kmeans", 5) not in results and n_samples >= 5:
        model_name = "kmeans"
        k = 5
        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init="auto"
        )
        labels = model.fit_predict(X)
        metric_value = compute_silhouette(X, labels)

        labels_series = pd.Series(
            labels,
            index=pivot.index,
            name="cluster"
        )
        results[(model_name, k)] = labels_series

        rows.append({
            "model": model_name,
            "k": int(k),
            "sample_size": n_samples,
            "metric": "silhouette",
            "metric_value": metric_value,
            "inertia": float(model.inertia_),
        })

    # -----------------------------------
    # 11) META-CLUSTERING on ONLY 3 FEATURES
    # -----------------------------------
    from sklearn.preprocessing import StandardScaler as _StdScaler

    meta_index = pivot.index

    abcxyz_codes, _ = pd.factorize(abcxyz_class.reindex(meta_index))
    demand_codes, _ = pd.factorize(pattern.reindex(meta_index))

    if ("kmeans", 5) not in results:
        raise ValueError("First-level KMeans k=5 missing. Ensure k_list includes 5.")
    kmeans5_labels = results[("kmeans", 5)].reindex(meta_index).astype(float)

    meta_features = pd.DataFrame({
        "ABCXYZ_feat": abcxyz_codes.astype(float),
        "DemandPattern_feat": demand_codes.astype(float),
        "KMeans_k5_feat": kmeans5_labels,
    }, index=meta_index)

    scaler_meta = _StdScaler()
    X_meta = scaler_meta.fit_transform(meta_features.values)

    for k_meta in [3, 4, 5, 6, 7, 8, 9, 10]:
        if k_meta < 2 or k_meta > len(meta_index):
            continue
        
        meta_model_name = "meta_kmeans"
        meta_model = KMeans(
            n_clusters=k_meta,
            random_state=42,
            n_init="auto"
        )
        meta_labels = meta_model.fit_predict(X_meta)

        metric_value_meta = compute_silhouette(X_meta, meta_labels)

        labels_series_meta = pd.Series(
            meta_labels,
            index=meta_index,
            name="cluster"
        )

        results[(meta_model_name, k_meta)] = labels_series_meta

        rows.append({
            "model": meta_model_name,
            "k": int(k_meta),
            "sample_size": len(meta_index),
            "metric": "silhouette",
            "metric_value": metric_value_meta,
        })

    # -----------------------------------
    # 12) MANUAL CLUSTERING MODEL (if provided)
    # -----------------------------------
    if manual_cluster_path is not None:
        import os
        if not os.path.exists(manual_cluster_path):
            print(f"[WARNING] manual_cluster_path not found: {manual_cluster_path}")
        else:
            print(f"\n[INFO] Loading manual clusters from: {manual_cluster_path}")
            df_clusters = pd.read_excel(manual_cluster_path)

            df_clusters = df_clusters.iloc[:, :2].copy()
            df_clusters.columns = ["Material Code", "cluster"]

            df_clusters["Material Code"] = df_clusters["Material Code"].astype(str)
            manual_map = (
                df_clusters
                .dropna(subset=["Material Code", "cluster"])
                .set_index("Material Code")["cluster"]
                .to_dict()
            )

            manual_labels_list = [
                manual_map.get(str(code), np.nan) for code in pivot.index
            ]
            manual_labels = pd.Series(
                manual_labels_list,
                index=pivot.index,
                name="cluster"
            )

            unique_manual = pd.unique(manual_labels.dropna())
            k_manual = len(unique_manual)

            if k_manual >= 2:
                metric_value_manual = compute_silhouette(X, manual_labels.values)

                results[("Manual", k_manual)] = manual_labels

                rows.append({
                    "model": "Manual",
                    "k": int(k_manual),
                    "sample_size": n_samples,
                    "metric": "silhouette",
                    "metric_value": metric_value_manual,
                })

                print(f"[INFO] Manual clustering added: k={k_manual}, silhouette={metric_value_manual:.3f}")
            else:
                print("[WARNING] Manual clustering has < 2 unique clusters or is empty; skipped in results.")

    # -----------------------------------
    # 13) Build summary_df
    # -----------------------------------
    summary_df = pd.DataFrame(rows).sort_values(["model", "k"]).reset_index(drop=True)

    # -----------------------------------
    # 14) Build ALL cluster label columns into df_aug
    # -----------------------------------
    df_aug = df.copy()

    for _, r in summary_df.iterrows():
        model_name = r["model"]
        k = r["k"]
        sil = r["metric_value"]

        if pd.isna(sil):
            col_name = f"{model_name}_k{k}_sil_NA"
        else:
            col_name = f"{model_name}_k{k}_sil_{sil:.3f}"

        labels_series = results.get((model_name, k))
        if labels_series is None:
            continue

        df_aug[col_name] = df_aug[SKU_COL].map(labels_series)

    seg_cols = ["ABC_Class", "XYZ_Class", "ABCXYZ_Class", "ADI", "CV", "CV2", "Demand_Pattern"]

    cluster_cols = [
        c for c in df_aug.columns
        if c not in df.columns and c not in seg_cols
    ]
    base_cols = [c for c in df_aug.columns if c not in seg_cols + cluster_cols]
    ordered_cols = base_cols + seg_cols + cluster_cols
    df_aug = df_aug[ordered_cols]

    # -----------------------------------
    # 15) Save detailed row-level file (ClusterV1_detail)
    # -----------------------------------
    cluster_detail_path = r"workspace/Clustering/ClusterV1_detail.xlsx"
    df_aug.to_excel(cluster_detail_path, index=False)
    print(f"\n[INFO] Detailed clustering output saved to: {cluster_detail_path}")

    # -----------------------------------
    # 16) Build SKU-level summary (ClusterV1)
    # -----------------------------------
    cols_to_remove = [
        "Date_Gregorian",
        "Hijri_Month_Main",
        "Hijri_Month_Secondary",
        "Vol (Kg)"
    ]
    df_sku = df_aug.drop(columns=cols_to_remove, errors="ignore")
    df_sku = df_sku.drop_duplicates(subset=[SKU_COL], keep="first")

    cluster_output_path = r"workspace/Clustering/ClusterV1.xlsx"
    df_sku.to_excel(cluster_output_path, index=False)
    print(f"[INFO] SKU-level clustering summary saved to: {cluster_output_path}")

    # -----------------------------------
    # 17) OPTIONAL: Build Cluster_Final, Cluster_Final_detail, Cluster_Final_Aggregated
    # -----------------------------------
    if final_model_name is not None and final_k is not None:
        key = (final_model_name, final_k)
        if key not in results:
            print(f"[WARNING] Requested final model {key} not found in results; "
                  f"Cluster_Final files will NOT be created.")
        else:
            final_labels = results[key]

            # Add Cluster_Final to df_aug and df_sku
            df_aug["Cluster_Final"] = df_aug[SKU_COL].map(final_labels)

            # recompute df_sku to include Cluster_Final
            df_sku = df_aug.drop(columns=cols_to_remove, errors="ignore")
            df_sku = df_sku.drop_duplicates(subset=[SKU_COL], keep="first")

            # SKU name column (robust)
            SKU_NAME_COL = "Sku Name" if "Sku Name" in df_aug.columns else SKU_COL

            # --- Cluster_Final.xlsx ---
            df_final = df_sku[[SKU_COL, SKU_NAME_COL, "Cluster_Final"]].copy()
            cluster_final_path = r"workspace/Clustering/Cluster_Final.xlsx"
            df_final.to_excel(cluster_final_path, index=False)
            print(f"[INFO] Final cluster summary saved to: {cluster_final_path}")

            # --- Cluster_Final_detail.xlsx ---
            detail_cols = [
                "Date_Gregorian",
                "Hijri_Month_Main",
                "Hijri_Month_Secondary",
                SKU_COL,
                SKU_NAME_COL,
                "Cluster_Final",
                "Vol (Kg)",
            ]
            detail_cols_existing = [c for c in detail_cols if c in df_aug.columns]
            df_final_detail = df_aug[detail_cols_existing].copy()

            cluster_final_detail_path = r"workspace/Clustering/Cluster_Final_detail.xlsx"
            df_final_detail.to_excel(cluster_final_detail_path, index=False)
            print(f"[INFO] Final detailed cluster file saved to: {cluster_final_detail_path}")

            # --- Cluster_Final_Aggregated.xlsx ---
            # Aggregate Vol (Kg) by Date_Gregorian (+ Hijri months if present) and Cluster_Final
            req_cols = ["Date_Gregorian", "Cluster_Final", "Vol (Kg)"]
            if all(c in df_final_detail.columns for c in req_cols):
                group_cols = ["Date_Gregorian"]
                # include Hijri months if they exist
                for c in ["Hijri_Month_Main", "Hijri_Month_Secondary"]:
                    if c in df_final_detail.columns:
                        group_cols.append(c)
                group_cols.append("Cluster_Final")

                df_agg = (
                    df_final_detail
                    .groupby(group_cols, as_index=False)["Vol (Kg)"]
                    .sum()
                    .sort_values(group_cols)
                )

                cluster_final_agg_path = r"workspace/Clustering/Cluster_Final_Aggregated.xlsx"
                df_agg.to_excel(cluster_final_agg_path, index=False)
                print(f"[INFO] Final aggregated cluster file saved to: {cluster_final_agg_path}")
            else:
                print("[WARNING] Could not create Cluster_Final_Aggregated "
                      "because required columns were missing.")

    # -----------------------------------
    # 18) Return for further use
    # -----------------------------------
    return df_aug, pivot, results, summary_df

## Cluster visualization

In [ ]:
def plot_cluster_figures(df_ts, cluster_labels):
    """
    Builds four sets of figures for clustered SKUs:

    1) RAW HISTORY PER CLUSTER
       - For each cluster: one figure with all SKUs in that cluster,
         showing historical sales time series (monthly) simultaneously.

    2) AGGREGATED HISTORY PER CLUSTER
       - For each cluster: one figure with a single line,
         where each point is the sum of Vol (Kg) of all SKUs in that cluster
         in that month (e.g. 2022-05).

    3) YEARLY MONTHLY PATTERN PER CLUSTER & SKU
       - Interactive figure per cluster:
         for a selected SKU, show monthly pattern split by year.

    4) DECOMPOSITION PER CLUSTER & SKU
       - For each cluster: three interactive figures (Trend, Seasonality, Residual)
         with a dropdown to switch between SKUs.
    """

    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    from statsmodels.tsa.seasonal import seasonal_decompose

    # -----------------------------------
    # Column names / config
    # -----------------------------------
    DATE_COL = "Date_Gregorian"
    SKU_COL = "Material Code"
    VOL_COL = "Vol (Kg)"
    SKU_NAME_COL = "Sku Name" if "Sku Name" in df_ts.columns else "Material Code"
    YEAR_COL = "year"
    MONTH_COL = "month"
    SEASONALITY = 12  # monthly data

    # Ensure time helper columns exist
    df_local = df_ts.copy()
    df_local[DATE_COL] = pd.to_datetime(df_local[DATE_COL])

    if YEAR_COL not in df_local.columns:
        df_local[YEAR_COL] = df_local[DATE_COL].dt.year
    if MONTH_COL not in df_local.columns:
        df_local[MONTH_COL] = df_local[DATE_COL].dt.month
    if "month_period" not in df_local.columns:
        df_local["month_period"] = df_local[DATE_COL].dt.to_period("M")

    # -----------------------------------
    # Normalize cluster_labels → DataFrame with [SKU_COL, "cluster"]
    # -----------------------------------
    if isinstance(cluster_labels, pd.Series):
        clustered_skus_local = cluster_labels.to_frame(name="cluster").reset_index()
        if clustered_skus_local.columns[0] != SKU_COL:
            clustered_skus_local.rename(
                columns={clustered_skus_local.columns[0]: SKU_COL},
                inplace=True
            )
    else:
        clustered_skus_local = cluster_labels.copy()
        if "cluster" not in clustered_skus_local.columns:
            raise ValueError("cluster_labels DataFrame must have a 'cluster' column.")

    # 🔴 IMPORTANT FIX:
    # Drop NaN clusters and convert all cluster labels to string
    clustered_skus_local = clustered_skus_local.dropna(subset=["cluster"]).copy()
    clustered_skus_local["cluster"] = clustered_skus_local["cluster"].astype(str)

    # ============================================================
    # 1) RAW HISTORY PER CLUSTER
    # ============================================================
    def _plot_cluster_raw_per_cluster(df_local_inner, clustered_skus_inner):
        df_plot = df_local_inner.merge(
            clustered_skus_inner[[SKU_COL, "cluster"]].drop_duplicates(),
            on=SKU_COL,
            how="inner",
        )

        # Fix SKU name column (if merge created _x/_y)
        name_candidates = [SKU_NAME_COL, f"{SKU_NAME_COL}_x", f"{SKU_NAME_COL}_y"]
        actual_name_col = None
        for cand in name_candidates:
            if cand in df_plot.columns:
                actual_name_col = cand
                break
        if actual_name_col is not None and actual_name_col != SKU_NAME_COL:
            df_plot[SKU_NAME_COL] = df_plot[actual_name_col]

        # cluster is already string; just sort
        clusters = sorted(df_plot["cluster"].unique())

        for c in clusters:
            cluster_df = df_plot[df_plot["cluster"] == c].copy()
            if cluster_df.empty:
                continue

            # Aggregate monthly per SKU
            cluster_df_grouped = (
                cluster_df
                .groupby([SKU_COL, DATE_COL], as_index=False)[VOL_COL]
                .sum()
                .sort_values(DATE_COL)
            )

            fig = go.Figure()
            sku_info = (
                cluster_df[[SKU_COL, SKU_NAME_COL]]
                .drop_duplicates()
                .sort_values(SKU_COL)
            )

            for _, row in sku_info.iterrows():
                code = row[SKU_COL]
                name = row.get(SKU_NAME_COL, "")
                label = f"{code} - {name}"

                sub = cluster_df_grouped[cluster_df_grouped[SKU_COL] == code]
                if sub.empty:
                    continue

                fig.add_trace(
                    go.Scatter(
                        x=sub[DATE_COL],
                        y=sub[VOL_COL],
                        mode="lines",
                        name=label,
                    )
                )

            fig.update_layout(
                title=f"Cluster {c} - Raw historical sales of all SKUs",
                xaxis_title="Date",
                yaxis_title="Sales Volume (Kg)",
                legend=dict(title="SKU"),
            )
            fig.show()

    # ============================================================
    # 2) AGGREGATED HISTORY PER CLUSTER: مجموع فروش کل کلاستر
    # ============================================================
    def _plot_cluster_aggregated_history(df_local_inner, clustered_skus_inner):
        """
        For each cluster:
            - Aggregate Vol (Kg) across all SKUs in that cluster per Date_Gregorian
            - Plot a single time series per cluster.
        """
        df_plot = df_local_inner.merge(
            clustered_skus_inner[[SKU_COL, "cluster"]].drop_duplicates(),
            on=SKU_COL,
            how="inner",
        )

        clusters = sorted(df_plot["cluster"].unique())

        for c in clusters:
            cluster_df = df_plot[df_plot["cluster"] == c].copy()
            if cluster_df.empty:
                continue

            # Aggregate monthly across all SKUs in the cluster
            agg_df = (
                cluster_df
                .groupby([DATE_COL], as_index=False)[VOL_COL]
                .sum()
                .sort_values(DATE_COL)
            )

            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=agg_df[DATE_COL],
                    y=agg_df[VOL_COL],
                    mode="lines+markers",
                    name=f"Cluster {c} - aggregated sales",
                )
            )

            fig.update_layout(
                title=f"Cluster {c} - Aggregated historical sales (sum of all SKUs)",
                xaxis_title="Date",
                yaxis_title="Total Sales Volume (Kg)",
                legend=dict(title=""),
            )
            fig.show()

    # ============================================================
    # 3) YEARLY MONTHLY PATTERN PER CLUSTER & SKU
    # ============================================================
    def _plot_cluster_sku_yearly_pattern(df_local_inner, clustered_skus_inner):
        df_plot = df_local_inner.merge(
            clustered_skus_inner[[SKU_COL, "cluster"]].drop_duplicates(),
            on=SKU_COL,
            how="inner",
        )

        # Fix SKU name column
        name_candidates = [SKU_NAME_COL, f"{SKU_NAME_COL}_x", f"{SKU_NAME_COL}_y"]
        actual_name_col = None
        for cand in name_candidates:
            if cand in df_plot.columns:
                actual_name_col = cand
                break
        if actual_name_col is not None and actual_name_col != SKU_NAME_COL:
            df_plot[SKU_NAME_COL] = df_plot[actual_name_col]

        clusters = sorted(df_plot["cluster"].unique())

        for c in clusters:
            cluster_df = df_plot[df_plot["cluster"] == c].copy()
            if cluster_df.empty:
                continue

            sku_info = cluster_df[[SKU_COL, SKU_NAME_COL]].drop_duplicates()
            sku_codes = sku_info[SKU_COL].tolist()

            code_to_label = {}
            for _, row in sku_info.iterrows():
                code = row[SKU_COL]
                name = row.get(SKU_NAME_COL, "")
                label = f"{code} - {name}"
                code_to_label[code] = label

            years = sorted(cluster_df[YEAR_COL].dropna().unique())
            if not years:
                continue

            fig = go.Figure()
            traces_for_sku = {code: [] for code in sku_codes}

            for code in sku_codes:
                sub = cluster_df[cluster_df[SKU_COL] == code].copy()
                if sub.empty:
                    continue

                grouped = (
                    sub.groupby([YEAR_COL, MONTH_COL])[VOL_COL]
                       .sum()
                       .reset_index()
                )

                for y in years:
                    g_y = grouped[grouped[YEAR_COL] == y].sort_values(MONTH_COL)
                    if g_y.empty:
                        continue
                    trace_name = f"{code_to_label[code]} | Year {y}"
                    trace = go.Scatter(
                        x=g_y[MONTH_COL],
                        y=g_y[VOL_COL],
                        mode="lines+markers",
                        name=trace_name,
                        visible=False
                    )
                    fig.add_trace(trace)
                    traces_for_sku[code].append(len(fig.data) - 1)

            if not fig.data:
                continue

            default_code = sku_codes[0]
            for idx in traces_for_sku[default_code]:
                fig.data[idx].visible = True

            buttons = []
            for code in sku_codes:
                visible = [False] * len(fig.data)
                for idx in traces_for_sku[code]:
                    visible[idx] = True

                label = code_to_label[code]
                buttons.append(
                    dict(
                        label=label,
                        method="update",
                        args=[
                            {"visible": visible},
                            {"title": f"Cluster {c} - {label} monthly pattern by year"}
                        ],
                    )
                )

            fig.update_layout(
                title=f"Cluster {c} - {code_to_label[default_code]} monthly pattern by year",
                xaxis_title="Month (1–12)",
                yaxis_title="Sales Volume (Kg)",
                xaxis=dict(tickmode="linear", dtick=1),
                updatemenus=[
                    dict(
                        buttons=buttons,
                        direction="down",
                        showactive=True,
                        x=1.05,
                        y=1,
                        xanchor="left",
                        yanchor="top",
                    )
                ],
                legend=dict(title="Year traces"),
            )

            fig.show()

    # ============================================================
    # 4) DECOMPOSITION (TREND / SEASONALITY / RESIDUAL)
    # ============================================================
    def _plot_cluster_sku_decomposition(df_local_inner, clustered_skus_inner):
        df_plot = df_local_inner.merge(
            clustered_skus_inner[[SKU_COL, "cluster"]].drop_duplicates(),
            on=SKU_COL,
            how="inner",
        )

        name_candidates = [SKU_NAME_COL, f"{SKU_NAME_COL}_x", f"{SKU_NAME_COL}_y"]
        actual_name_col = None
        for cand in name_candidates:
            if cand in df_plot.columns:
                actual_name_col = cand
                break
        if actual_name_col is not None and actual_name_col != SKU_NAME_COL:
            df_plot[SKU_NAME_COL] = df_plot[actual_name_col]

        clusters = sorted(df_plot["cluster"].unique())

        for c in clusters:
            cluster_df = df_plot[df_plot["cluster"] == c].copy()
            if cluster_df.empty:
                continue

            sku_info = cluster_df[[SKU_COL, SKU_NAME_COL]].drop_duplicates()
            sku_codes = sku_info[SKU_COL].tolist()

            code_to_label = {}
            for _, row in sku_info.iterrows():
                code = row[SKU_COL]
                name = row.get(SKU_NAME_COL, "")
                label = f"{code} - {name}"
                code_to_label[code] = label

            decomp_map = {}  # sku_code -> (dates, trend, seasonal, resid)

            global_min_period = df_local_inner["month_period"].min()
            global_max_period = df_local_inner["month_period"].max()
            all_months = pd.period_range(global_min_period, global_max_period, freq="M")

            for code in sku_codes:
                sub = cluster_df[cluster_df[SKU_COL] == code].copy()
                if sub.empty:
                    continue

                s = (
                    sub.set_index("month_period")[VOL_COL]
                       .reindex(all_months)
                       .fillna(0.0)
                )
                s = s.to_timestamp(how="start")
                values = s.values

                if np.any(values <= 0):
                    model = "add"
                else:
                    model = "mul"

                try:
                    res = seasonal_decompose(
                        s,
                        model=model,
                        period=SEASONALITY,
                        two_sided=False,
                    )
                    trend = res.trend
                    seasonal = res.seasonal
                    resid = res.resid
                    dates = s.index

                    decomp_map[code] = (dates, trend, seasonal, resid)
                except Exception as e:
                    print(f"Warning: decomposition failed for SKU {code} in cluster {c}: {e}")

            if not decomp_map:
                continue

            # TREND
            trend_fig = go.Figure()
            trend_traces_for_sku = {code: [] for code in decomp_map.keys()}
            for code, (dates, trend, seasonal, resid) in decomp_map.items():
                trend_fig.add_trace(
                    go.Scatter(
                        x=dates,
                        y=trend,
                        mode="lines",
                        name=code_to_label.get(code, str(code)),
                        visible=False,
                    )
                )
                trend_traces_for_sku[code].append(len(trend_fig.data) - 1)

            # SEASONALITY
            seas_fig = go.Figure()
            seas_traces_for_sku = {code: [] for code in decomp_map.keys()}
            for code, (dates, trend, seasonal, resid) in decomp_map.items():
                seas_fig.add_trace(
                    go.Scatter(
                        x=dates,
                        y=seasonal,
                        mode="lines",
                        name=code_to_label.get(code, str(code)),
                        visible=False,
                    )
                )
                seas_traces_for_sku[code].append(len(seas_fig.data) - 1)

            # RESIDUAL
            resid_fig = go.Figure()
            resid_traces_for_sku = {code: [] for code in decomp_map.keys()}
            for code, (dates, trend, seasonal, resid) in decomp_map.items():
                resid_fig.add_trace(
                    go.Scatter(
                        x=dates,
                        y=resid,
                        mode="lines",
                        name=code_to_label.get(code, str(code)),
                        visible=False,
                    )
                )
                resid_traces_for_sku[code].append(len(resid_fig.data) - 1)

            default_code = list(decomp_map.keys())[0]
            for idx in trend_traces_for_sku[default_code]:
                trend_fig.data[idx].visible = True
            for idx in seas_traces_for_sku[default_code]:
                seas_fig.data[idx].visible = True
            for idx in resid_traces_for_sku[default_code]:
                resid_fig.data[idx].visible = True

            buttons_trend = []
            buttons_seas = []
            buttons_resid = []

            for code in decomp_map.keys():
                label = code_to_label.get(code, str(code))

                vis_trend = [False] * len(trend_fig.data)
                for idx in trend_traces_for_sku[code]:
                    vis_trend[idx] = True
                buttons_trend.append(
                    dict(
                        label=label,
                        method="update",
                        args=[
                            {"visible": vis_trend},
                            {"title": f"Cluster {c} - TREND - {label}"}
                        ],
                    )
                )

            for code in decomp_map.keys():
                label = code_to_label.get(code, str(code))

                vis_seas = [False] * len(seas_fig.data)
                for idx in seas_traces_for_sku[code]:
                    vis_seas[idx] = True
                buttons_seas.append(
                    dict(
                        label=label,
                        method="update",
                        args=[
                            {"visible": vis_seas},
                            {"title": f"Cluster {c} - SEASONALITY - {label}"}
                        ],
                    )
                )

            for code in decomp_map.keys():
                label = code_to_label.get(code, str(code))

                vis_resid = [False] * len(resid_fig.data)
                for idx in resid_traces_for_sku[code]:
                    vis_resid[idx] = True
                buttons_resid.append(
                    dict(
                        label=label,
                        method="update",
                        args=[
                            {"visible": vis_resid},
                            {"title": f"Cluster {c} - RESIDUAL - {label}"}
                        ],
                    )
                )

            trend_fig.update_layout(
                title=f"Cluster {c} - TREND - {code_to_label.get(default_code, str(default_code))}",
                xaxis_title="Time",
                yaxis_title="Trend component",
                updatemenus=[
                    dict(
                        buttons=buttons_trend,
                        direction="down",
                        showactive=True,
                        x=1.05,
                        y=1,
                        xanchor="left",
                        yanchor="top",
                    )
                ],
            )

            seas_fig.update_layout(
                title=f"Cluster {c} - SEASONALITY - {code_to_label.get(default_code, str(default_code))}",
                xaxis_title="Time",
                yaxis_title="Seasonal component",
                updatemenus=[
                    dict(
                        buttons=buttons_seas,
                        direction="down",
                        showactive=True,
                        x=1.05,
                        y=1,
                        xanchor="left",
                        yanchor="top",
                    )
                ],
            )

            resid_fig.update_layout(
                title=f"Cluster {c} - RESIDUAL - {code_to_label.get(default_code, str(default_code))}",
                xaxis_title="Time",
                yaxis_title="Residual component",
                updatemenus=[
                    dict(
                        buttons=buttons_resid,
                        direction="down",
                        showactive=True,
                        x=1.05,
                        y=1,
                        xanchor="left",
                        yanchor="top",
                    )
                ],
            )

            trend_fig.show()
            seas_fig.show()
            resid_fig.show()

    # -----------------------------------
    # Call four helper blocks
    # -----------------------------------
    _plot_cluster_raw_per_cluster(df_local, clustered_skus_local)
    _plot_cluster_aggregated_history(df_local, clustered_skus_local)
    _plot_cluster_sku_yearly_pattern(df_local, clustered_skus_local)
    _plot_cluster_sku_decomposition(df_local, clustered_skus_local)

## Clustering wrapper (legacy)

In [ ]:
def run_full_clustering_pipeline(
    final_model_name,
    final_k,
    raw_input_path=r"workspace/Input/monthly_sales.xlsx",
    hijri_output_path=r"workspace/Input/input.xlsx",
    manual_cluster_path=r"workspace/Clustering/ClusterManual_Plot.xlsx",
    do_plot=True,
):
    """
    Runs the full clustering pipeline end-to-end.

    Parameters
    ----------
    final_model_name : str
        The name of the clustering model you want to use
        (e.g., 'kmeans', 'agglo', 'gmm', 'hier_ts',
         'meta_kmeans', 'ABC', 'XYZ', 'ABCXYZ',
         'DemandPattern', 'Manual').

    final_k : int
        The number of clusters for the model you selected.
        Must match a k produced by cluster_time_series_data().

    raw_input_path : str
        Path to the base time-series input for clustering
        (default: your original monthly_sales.xlsx).

    hijri_output_path : str
        Path where the detailed clustering+Hijri output will be written
        (default: your original input.xlsx).

    manual_cluster_path : str or None
        Optional path to manual cluster mapping file.

    do_plot : bool
        If True  → call plot_cluster_figures for the chosen model/k.
        If False → skip plotting (useful when called from modeling pipeline).
    """

    import pandas as pd

    # -----------------------------------
    # Clustering configurations
    # -----------------------------------
    k_list = (3, 4, 5, 6, 7, 8, 9, 10)
    models = ("kmeans", "agglo", "gmm")

    # -----------------------------------
    # 1) Run clustering
    #    (pass final_model_name & final_k so Cluster_Final files are created correctly)
    # -----------------------------------
    df_ts, pivot, results, summary_df = cluster_time_series_data(
        raw_input_path=raw_input_path,
        hijri_output_path=hijri_output_path,
        k_list=k_list,
        models=models,
        manual_cluster_path=manual_cluster_path,
        final_model_name=final_model_name,   # <-- IMPORTANT
        final_k=final_k,                     # <-- IMPORTANT
    )

    print("\n=== Clustering Summary ===")
    print(summary_df.to_string(index=False))

    # -----------------------------------
    # 2) Validate model + k against what was actually produced
    # -----------------------------------
    if final_model_name not in summary_df["model"].unique():
        raise ValueError(
            f"Model '{final_model_name}' does not exist. "
            f"Valid models are: {sorted(summary_df['model'].unique())}"
        )

    ks_for_model = sorted(
        summary_df.loc[summary_df["model"] == final_model_name, "k"].unique()
    )
    if final_k not in ks_for_model:
        raise ValueError(
            f"k={final_k} is not valid for model '{final_model_name}'. "
            f"Valid ks: {ks_for_model}"
        )

    key = (final_model_name, final_k)
    if key not in results:
        raise ValueError(f"No clustering result found for model={final_model_name}, k={final_k}")

    cluster_labels = results[key]
    print(f"\n[INFO] Using FINAL clustering → model={final_model_name}, k={final_k}")

    # -----------------------------------
    # 3) Plot figures for the chosen clustering (optional)
    # -----------------------------------
    if do_plot:
        plot_cluster_figures(df_ts, cluster_labels)
        print("\n[INFO] Plotting completed.")
    else:
        print("\n[INFO] Plotting skipped (do_plot=False).")

    return df_ts, pivot, results, summary_df

## Volume normalization

In [ ]:
import pandas as pd
import numpy as np

def normalize_volumes(
    data_type,
    method,
    scaling_method,
    output_path,
    raw_input_path=r"workspace/Input/input.xlsx",
    cluster_input_path=r"workspace/Clustering/Cluster_Final_detail.xlsx",
):
    """
    Normalize Vol (Kg) either on RAW or CLUSTERED input.

    Parameters
    ----------
    data_type : str
        "Raw"      -> read from raw_input_path
        "Cluster"  -> read from cluster_input_path

    method : str
        Normalization scheme:
        - "per_sku", "sku", "each_material", "Normalize EACH MATERIAL CODE Across TIME"
            -> normalize EACH MATERIAL CODE across TIME
        - "per_month", "month", "Normalize MATERIAL CODES WITHIN EACH MONTH"
            -> normalize MATERIAL CODES WITHIN EACH MONTH & YEAR

    scaling_method : str
        How to scale values:
        - "standard"  -> StandardScaler: (x - mean) / std
        - "minmax"    -> Min–Max: (x - min) / (max - min)
        - "robust"    -> RobustScaler: (x - median) / IQR
        - "log1p"     -> log(1 + x)  [values clipped at 0 if negative]

    output_path : str
        Full path for saving the normalized file.

    raw_input_path : str
        Path for RAW input (default: workspace/Input/input.xlsx)

    cluster_input_path : str
        Path for CLUSTERED input (default: workspace/Clustering/Cluster_Final_detail.xlsx)


    Output
    ------
    - Writes an Excel file to `output_path` with the SAME columns as input,
      but `Vol (Kg)` replaced by normalized values.
    - Returns the resulting DataFrame.
    """

    # ------------------------------------------------------
    # 1) Select input path based on data_type
    # ------------------------------------------------------
    data_type_clean = str(data_type).strip().lower()
    if data_type_clean in ["raw", "r"]:
        input_path = raw_input_path
    elif data_type_clean in ["cluster", "clustered", "c"]:
        input_path = cluster_input_path
    else:
        raise ValueError("data_type must be 'Raw' or 'Cluster'.")

    # ------------------------------------------------------
    # 2) Read input file
    # ------------------------------------------------------
    df = pd.read_excel(input_path)

    # Column names
    VOL_COL = "Vol (Kg)"
    SKU_COL = "Material Code"
    DATE_COL = "Date_Gregorian"

    if VOL_COL not in df.columns:
        raise ValueError(f"Column '{VOL_COL}' not found in the input file.")
    if SKU_COL not in df.columns:
        raise ValueError(f"Column '{SKU_COL}' not found in the input file.")
    if DATE_COL not in df.columns:
        raise ValueError(f"Column '{DATE_COL}' not found in the input file.")

    # Ensure correct dtypes
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df[SKU_COL] = df[SKU_COL].astype(str)
    df[VOL_COL] = pd.to_numeric(df[VOL_COL], errors="coerce").fillna(0.0)

    # ------------------------------------------------------
    # 3) Interpret normalization "method"
    # ------------------------------------------------------
    method_clean = str(method).strip().lower()
    if (
        "each" in method_clean
        or "material" in method_clean
        or "sku" in method_clean
        or method_clean in ["per_sku", "sku"]
    ):
        normalize_mode = "per_sku"       # Normalize EACH MATERIAL CODE across TIME
    elif (
        "month" in method_clean
        or method_clean in ["per_month"]
    ):
        normalize_mode = "per_month"     # Normalize MATERIAL CODES within EACH MONTH
    else:
        raise ValueError(
            "Unknown method. Use something like "
            "'Normalize EACH MATERIAL CODE Across TIME' or 'Normalize MATERIAL CODES WITHIN EACH MONTH', "
            "or the short forms 'per_sku', 'per_month'."
        )

    # ------------------------------------------------------
    # 4) Interpret scaling_method
    # ------------------------------------------------------
    scaling_clean = str(scaling_method).strip().lower()
    valid_scalers = ["standard", "minmax", "robust", "log1p"]
    if scaling_clean not in valid_scalers:
        raise ValueError(f"scaling_method must be one of {valid_scalers}")

    # ------------------------------------------------------
    # 5) Define scaling function for a pandas Series
    # ------------------------------------------------------
    def scale_series(s: pd.Series) -> pd.Series:
        """Apply the chosen scaling to a 1D Series."""
        x = s.astype(float).values

        if scaling_clean == "log1p":
            # Clip negatives (if any) to 0 before log1p
            x = np.clip(x, a_min=0, a_max=None)
            x_scaled = np.log1p(x)
            return pd.Series(x_scaled, index=s.index)

        # Other scalers use distribution of the group
        if scaling_clean == "standard":
            mean = np.mean(x)
            std = np.std(x)
            if std == 0:
                x_scaled = np.zeros_like(x)
            else:
                x_scaled = (x - mean) / std

        elif scaling_clean == "minmax":
            x_min = np.min(x)
            x_max = np.max(x)
            if x_max == x_min:
                x_scaled = np.zeros_like(x)
            else:
                x_scaled = (x - x_min) / (x_max - x_min)

        elif scaling_clean == "robust":
            median = np.median(x)
            q1 = np.quantile(x, 0.25)
            q3 = np.quantile(x, 0.75)
            iqr = q3 - q1
            if iqr == 0:
                x_scaled = np.zeros_like(x)
            else:
                x_scaled = (x - median) / iqr

        else:
            # Should never hit here due to earlier check
            x_scaled = x

        return pd.Series(x_scaled, index=s.index)

    # ------------------------------------------------------
    # 6) Apply scaling according to normalize_mode
    # ------------------------------------------------------
    if normalize_mode == "per_sku":
        # Normalize EACH MATERIAL CODE across TIME
        df[VOL_COL] = df.groupby(SKU_COL)[VOL_COL].transform(scale_series)

    elif normalize_mode == "per_month":
        # Normalize MATERIAL CODES WITHIN EACH MONTH & YEAR
        df[VOL_COL] = df.groupby(DATE_COL)[VOL_COL].transform(scale_series)

    # ------------------------------------------------------
    # 7) Save and return
    # ------------------------------------------------------
    df.to_excel(output_path, index=False)
    print(f"[INFO] Normalized data saved to: {output_path}")
    print(f"[INFO] Mode: {normalize_mode}, Scaler: {scaling_clean}, Source: {input_path}")

    return df

## Normalization and visualization wrapper

In [ ]:
import pandas as pd

def run_normalization_and_plot(
    data_type,
    method,
    scaling_method,
    output_path,
    raw_input_path=r"workspace/Input/input.xlsx",
    cluster_input_path=r"workspace/Clustering/Cluster_Final_detail.xlsx",
    use_clustering=False,
    final_model_name=None,
    final_k=None,
):
    """
    Wrapper to:
      1) (Optionally) re-run clustering via run_full_clustering_pipeline
         to update Cluster_Final in Cluster_Final_detail.xlsx
      2) Normalize Vol (Kg) according to your choices
      3) Plot all four cluster figures on the normalized data

    Parameters
    ----------
    data_type : str
        "Raw"      -> read from raw_input_path
        "Cluster"  -> read from cluster_input_path

    method : str
        Normalization scheme:
        - "per_sku" or "Normalize EACH MATERIAL CODE Across TIME"
        - "per_month" or "Normalize MATERIAL CODES WITHIN EACH MONTH"

    scaling_method : str
        "standard" | "minmax" | "robust" | "log1p"

    output_path : str
        Path to save the normalized file (Excel).

    raw_input_path : str
        Path for RAW input (default: workspace/Input/input.xlsx)

    cluster_input_path : str
        Path for CLUSTERED input AND cluster labels
        (default: workspace/Clustering/Cluster_Final_detail.xlsx)
        - Column: "Material Code"
        - Column: "Cluster_Final"
        - Column: "Vol (Kg)"

    use_clustering : bool
        If True and data_type == "Cluster":
            call run_full_clustering_pipeline(final_model_name=..., final_k=...)
            BEFORE normalization to update Cluster_Final.

    final_model_name : str or None
        The chosen final clustering model name for run_full_clustering_pipeline,
        e.g. "kmeans", "agglo", "gmm". Used only if use_clustering=True.

    final_k : int or None
        The chosen k for the final clustering model. Used only if use_clustering=True.
    """

    data_type_clean = str(data_type).strip().lower()

    # -----------------------------------
    # 0) (Optional) Re-run clustering to update Cluster_Final
    # -----------------------------------
    if use_clustering and data_type_clean in ["cluster", "clustered", "c"]:
        if final_model_name is None or final_k is None:
            raise ValueError(
                "When use_clustering=True, you must provide final_model_name and final_k."
            )

        print("[INFO] Re-running clustering via run_full_clustering_pipeline...")
        # We assume run_full_clustering_pipeline is already defined in the notebook
        # and writes/updates Cluster_Final_detail.xlsx (cluster_input_path).
        run_full_clustering_pipeline(
            final_model_name=final_model_name,
            final_k=final_k,
        )
        print("[INFO] Clustering pipeline finished. Cluster_Final updated.")

    # -----------------------------------
    # 1) Normalize using the helper function (on updated / existing data)
    # -----------------------------------
    df_norm = normalize_volumes(
        data_type=data_type,
        method=method,
        scaling_method=scaling_method,
        output_path=output_path,
        raw_input_path=raw_input_path,
        cluster_input_path=cluster_input_path,
    )

    # -----------------------------------
    # 2) Load cluster labels
    #    - If data_type == "Cluster": take Cluster_Final directly from df_norm
    #    - If data_type == "Raw": read Cluster_Final from Cluster_Final_detail.xlsx
    # -----------------------------------
    if data_type_clean in ["cluster", "clustered", "c"]:
        # We rely on column names, not positions:
        if "Material Code" not in df_norm.columns:
            raise ValueError("Column 'Material Code' not found in normalized cluster file.")
        if "Cluster_Final" not in df_norm.columns:
            raise ValueError("Column 'Cluster_Final' not found in normalized cluster file.")

        df_clusters_local = df_norm[["Material Code", "Cluster_Final"]].dropna()

    else:  # data_type == "Raw"
        df_clusters = pd.read_excel(cluster_input_path)

        if "Material Code" not in df_clusters.columns:
            raise ValueError("Column 'Material Code' not found in Cluster_Final_detail.xlsx")
        if "Cluster_Final" not in df_clusters.columns:
            raise ValueError("Column 'Cluster_Final' not found in Cluster_Final_detail.xlsx")

        df_clusters_local = df_clusters[["Material Code", "Cluster_Final"]].dropna()

    # Ensure types match
    df_clusters_local["Material Code"] = df_clusters_local["Material Code"].astype(str)
    df_norm["Material Code"] = df_norm["Material Code"].astype(str)

    # Build cluster_labels Series: index = Material Code, values = Cluster_Final
    cluster_labels = (
        df_clusters_local
        .set_index("Material Code")["Cluster_Final"]
    )

    print("[INFO] Sample of cluster labels used for plotting:")
    print(cluster_labels.head())

    # -----------------------------------
    # 3) Plot all 4 sets of figures on NORMALIZED data
    # -----------------------------------
    plot_cluster_figures(df_norm, cluster_labels)

    return df_norm

## Modeling input orchestration

In [ ]:
import pandas as pd

def run_full_clustering_pipeline(final_model_name, final_k):
    """
    Runs the clustering pipeline (NO PLOTTING).

    - Builds inputv2 with Hijri columns from monthly_sales.xlsx.
    - Runs all clustering models via cluster_time_series_data.
    - Uses final_model_name & final_k inside cluster_time_series_data
      to create Cluster_Final.*, Cluster_Final_detail.*, etc.
    - Returns df_ts, pivot, results, summary_df for optional inspection.
    """

    # -----------------------------------
    # 0) Paths (change if needed)
    # -----------------------------------
    raw_input_path = r"workspace/Input/monthly_sales.xlsx"
    hijri_output_path = r"workspace/Input/inputv2.xlsx"
    manual_cluster_path = r"workspace/Clustering/ClusterManual_Plot.xlsx"

    # You can adjust k_list or models here if you want
    k_list = (3, 4, 5, 6, 7, 8, 9, 10)
    models = ("kmeans", "agglo", "gmm")

    # -----------------------------------
    # 1) Run clustering (NO plotting here)
    # -----------------------------------
    df_ts, pivot, results, summary_df = cluster_time_series_data(
        raw_input_path=raw_input_path,
        hijri_output_path=hijri_output_path,
        k_list=k_list,
        models=models,
        manual_cluster_path=manual_cluster_path,
        final_model_name=final_model_name,
        final_k=final_k,
    )

    print("\n=== Clustering Summary ===")
    print(summary_df.to_string(index=False))

    # This function NO LONGER plots.
    # Plotting is done at the end of the pipeline on df_prepared.
    return df_ts, pivot, results, summary_df


def prepare_modeling_data(
    pipeline_mode,
    normalization_method=None,
    scaling_method=None,
    final_model_name=None,
    final_k=None,
    base_raw_path=r"workspace/Input/inputv2.xlsx",
    cluster_detail_path=r"workspace/Clustering/Cluster_Final_detail.xlsx",
    temp_normalized_raw_path=r"workspace/Input/inputv2_normalized.xlsx",
    temp_normalized_cluster_path=r"workspace/Clustering/Cluster_Final_detail_normalized.xlsx",
    output_path=None,
):
    """
    Unified data preparation function to be used before modeling.

    Modes
    -----
    1) 'raw'
       - Use prepared raw data from base_raw_path.
       - No normalization, no clustering.
       - Cluster = 1.

    2) 'normalized'
       - Normalize RAW data via normalize_volumes(data_type='Raw').
       - No clustering (Cluster = 1).

    3) 'cluster_only'
       - Run clustering via run_full_clustering_pipeline on raw data.
       - Read Cluster_Final_detail and use Cluster_Final as Cluster.
       - Vol (Kg) raw.

    4) 'cluster_then_normalize'
       - Run clustering via run_full_clustering_pipeline on raw data.
       - Normalize CLUSTERED data via normalize_volumes(data_type='Cluster').
       - Cluster = Cluster_Final.

    5) 'normalize_then_cluster'
       - Normalize RAW data via normalize_volumes(data_type='Raw').
       - Run clustering via run_full_clustering_pipeline on raw data.
       - Map Cluster_Final from Cluster_Final.xlsx to normalized data by Material Code.

    Returns
    -------
    df_final : DataFrame with columns:
        ['Date_Gregorian','Hijri_Month_Main','Hijri_Month_Secondary',
         'Material Code','Sku Name','Vol (Kg)','Cluster']
    """

    def _select_final_columns(df):
        needed = [
            "Date_Gregorian",
            "Hijri_Month_Main",
            "Hijri_Month_Secondary",
            "Material Code",
            "Sku Name",
            "Vol (Kg)",
            "Cluster",
        ]
        existing = [c for c in needed if c in df.columns]
        df = df[existing].copy()
        missing = [c for c in needed if c not in df.columns]
        if missing:
            raise ValueError(f"The following required columns are missing in the prepared data: {missing}")
        return df

    # -----------------------------
    # 1) RAW MODE
    # -----------------------------
    if pipeline_mode == "raw":
        df = pd.read_excel(base_raw_path)
        df["Cluster"] = 1
        df_final = _select_final_columns(df)

    # -----------------------------
    # 2) NORMALIZED MODE
    # -----------------------------
    elif pipeline_mode == "normalized":
        if normalization_method is None or scaling_method is None:
            raise ValueError("normalization_method and scaling_method must be provided for 'normalized' mode.")

        # normalize_volumes(data_type, method, scaling_method, output_path, raw_input_path, cluster_input_path)
        normalize_volumes(
            "Raw",
            normalization_method,
            scaling_method,
            temp_normalized_raw_path,
            base_raw_path,
            None,
        )

        df = pd.read_excel(temp_normalized_raw_path)
        df["Cluster"] = 1
        df_final = _select_final_columns(df)

    # -----------------------------
    # 3) CLUSTER-ONLY MODE
    # -----------------------------
    elif pipeline_mode == "cluster_only":
        if final_model_name is None or final_k is None:
            raise ValueError("final_model_name and final_k must be provided for 'cluster_only' mode.")

        # run clustering (no plotting)
        run_full_clustering_pipeline(final_model_name, final_k)

        df = pd.read_excel(cluster_detail_path)

        if "Cluster_Final" not in df.columns:
            raise ValueError("Expected 'Cluster_Final' column in Cluster_Final_detail file.")
        df["Cluster"] = df["Cluster_Final"]

        df_final = _select_final_columns(df)

    # -----------------------------
    # 4) CLUSTER THEN NORMALIZE MODE
    # -----------------------------
    elif pipeline_mode == "cluster_then_normalize":
        if final_model_name is None or final_k is None:
            raise ValueError("final_model_name and final_k must be provided for 'cluster_then_normalize' mode.")
        if normalization_method is None or scaling_method is None:
            raise ValueError("normalization_method and scaling_method must be provided for 'cluster_then_normalize' mode.")

        # Step 1: run clustering on raw data (no plotting)
        run_full_clustering_pipeline(final_model_name, final_k)

        # Step 2: normalize clustered data (Cluster_Final_detail.xlsx)
        normalize_volumes(
            "Cluster",
            normalization_method,
            scaling_method,
            temp_normalized_cluster_path,
            None,
            cluster_detail_path,
        )

        df = pd.read_excel(temp_normalized_cluster_path)

        if "Cluster_Final" not in df.columns:
            raise ValueError("Expected 'Cluster_Final' column in normalized clustered file.")
        df["Cluster"] = df["Cluster_Final"]

        df_final = _select_final_columns(df)

    # -----------------------------
    # 5) NORMALIZE THEN CLUSTER MODE
    # -----------------------------
    elif pipeline_mode == "normalize_then_cluster":
        if final_model_name is None or final_k is None:
            raise ValueError("final_model_name and final_k must be provided for 'normalize_then_cluster' mode.")
        if normalization_method is None or scaling_method is None:
            raise ValueError("normalization_method and scaling_method must be provided for 'normalize_then_cluster' mode.")

        # Step 1: normalize RAW data → temp_normalized_raw_path
        normalize_volumes(
            "Raw",
            normalization_method,
            scaling_method,
            temp_normalized_raw_path,
            base_raw_path,
            None,
        )

        # Step 2: run clustering pipeline on raw data (no plotting)
        run_full_clustering_pipeline(final_model_name, final_k)

        # Load normalized data
        df_norm = pd.read_excel(temp_normalized_raw_path)

        # Load final cluster summary
        cluster_final_path = r"workspace/Clustering/Cluster_Final.xlsx"
        df_final_clusters = pd.read_excel(cluster_final_path)

        if "Cluster_Final" not in df_final_clusters.columns:
            raise ValueError("Expected 'Cluster_Final' column in Cluster_Final.xlsx.")

        df_final_clusters["Material Code"] = df_final_clusters["Material Code"].astype(str)
        df_norm["Material Code"] = df_norm["Material Code"].astype(str)

        cluster_map = (
            df_final_clusters
            .dropna(subset=["Material Code", "Cluster_Final"])
            .set_index("Material Code")["Cluster_Final"]
            .to_dict()
        )

        df_norm["Cluster"] = df_norm["Material Code"].map(cluster_map)

        if df_norm["Cluster"].isna().any():
            missing_count = df_norm["Cluster"].isna().sum()
            print(f"[WARN] {missing_count} rows have no Cluster assigned in 'normalize_then_cluster' mode.")

        df_final = _select_final_columns(df_norm)

    else:
        raise ValueError(
            f"Unknown pipeline_mode='{pipeline_mode}'. "
            "Must be one of: 'raw','normalized','cluster_only','cluster_then_normalize','normalize_then_cluster'."
        )

    if output_path is not None:
        df_final.to_excel(output_path, index=False)
        print(f"[INFO] Final prepared dataset saved to: {output_path}")

    return df_final

## Example configuration and execution

In [ ]:
# --- Configuration ---
pipeline_mode_choice = "cluster_only"   # 'raw','normalized','cluster_only','cluster_then_normalize','normalize_then_cluster'

normalization_method_choice = "Normalize EACH MATERIAL CODE Across TIME"  # or your own label
scaling_method_choice = "log1p"                # 'standard','minmax','robust','log1p'

final_model_choice = "meta_kmeans"                     # e.g. 'kmeans','agglo','gmm','hier_ts','meta_kmeans','ABC','XYZ','ABCXYZ','DemandPattern','Manual'
final_k_choice = 5

base_raw_path = r"workspace/Input/inputv2.xlsx"
cluster_detail_path = r"workspace/Clustering/Cluster_Final_detail.xlsx"
temp_normalized_raw_path = r"workspace/Input/inputv2_normalized.xlsx"
temp_normalized_cluster_path = r"workspace/Clustering/Cluster_Final_detail_normalized.xlsx"
output_path = r"workspace/Clustering/Modeling_Input.xlsx"

# --- 1) Run the unified data-preparation pipeline ---
df_prepared = prepare_modeling_data(
    pipeline_mode=pipeline_mode_choice,
    normalization_method=normalization_method_choice,
    scaling_method=scaling_method_choice,
    final_model_name=final_model_choice,
    final_k=final_k_choice,
    base_raw_path=base_raw_path,
    cluster_detail_path=cluster_detail_path,
    temp_normalized_raw_path=temp_normalized_raw_path,
    temp_normalized_cluster_path=temp_normalized_cluster_path,
    output_path=output_path,
)

print("\n[INFO] Sample of prepared data:")
print(df_prepared.head())

# --- 2) Plot ONLY at the end, using the FINAL (normalized/clustered) data ---
# We plot if clustering is actually used in the chosen pipeline
cluster_modes = ("cluster_only", "cluster_then_normalize", "normalize_then_cluster")

if pipeline_mode_choice in cluster_modes:
    print("\n[INFO] Building cluster labels from prepared data for plotting ...")

    df_plot = df_prepared.copy()
    df_plot["Material Code"] = df_plot["Material Code"].astype(str)

    # Build cluster_labels Series: index=Material Code, value=Cluster
    cluster_labels = (
        df_plot[["Material Code", "Cluster"]]
        .dropna(subset=["Material Code", "Cluster"])
        .drop_duplicates(subset=["Material Code"])
        .set_index("Material Code")["Cluster"]
    )

    print("[INFO] Plotting normalized/clustered data based on prepared dataset ...")
    plot_cluster_figures(df_plot, cluster_labels)
else:
    print("\n[INFO] Pipeline mode has no clustering → no cluster plots generated.")

## Statistical pre-estimation

In [ ]:
def pre_estimate_modeling_data(
    input_path=r"workspace/Clustering/Modeling_Input.xlsx",
    seasonality=None,
    output_path=r"workspace/PreEstimation/PreEstimation_Output.xlsx",
):
    """
    Completely stand-alone PRE-ESTIMATION pipeline.
    No external custom functions (e.g., mohKruskalSeasTest) are imported.

    The following outputs are created per SKU:
        - DumOut
        - datatype
        - optdiff
        - optDiff
        - finalSeasKW
    """

    import numpy as np
    import pandas as pd
    import warnings
    import statsmodels.api as sm
    from statsmodels.tsa.stattools import bds as bdstst, adfuller
    from scipy.stats import kruskal

    # ---------------------------------------------------------
    # INTERNAL REPLACEMENT of mohKruskalSeasTest  (FULL LOGIC)
    # ---------------------------------------------------------
    def local_mohKruskalSeasTest(y, season):
        """
        Re-implemented internally.
        Equivalent to mohKruskalSeasTest(y, season).

        Procedure:
        - Split y into "season" groups
        - Apply Kruskal-Wallis test
        - Return (KW-statistic, p-value)
        """

        y = np.asarray(y)
        n = len(y)

        if season <= 1 or season >= n:
            # No seasonal structure possible
            return 0, 1.0

        # Build seasonal groups
        groups = []
        for s in range(season):
            group_vals = y[s::season]
            if len(group_vals) > 0:
                groups.append(group_vals)

        # If fewer than 2 groups, no test
        if len(groups) < 2:
            return 0, 1.0

        try:
            KW, p = kruskal(*groups, nan_policy="omit")
        except Exception:
            KW, p = 0, 1.0

        return KW, p

    # ---------------------------------------------------------
    # INTERNAL REPLACEMENT of SEASONAL DIFFERENCING TEST
    # (Simplified OCSB alternative: variance ratio test)
    # ---------------------------------------------------------
    def local_seasonal_diff_test(y, seas):
        """
        Replacement for OCSB seasonal differencing test.
        Rule:
            If variance of seasonal difference < variance of first difference,
            then seasonal differencing improves stationarity → optDiff = 1
        """

        y = np.asarray(y, dtype=float)
        if seas is None or seas <= 1:
            return 0

        if len(y) <= seas + 2:
            return 0

        # regular diff
        d1 = np.diff(y, 1)
        # seasonal diff
        ds = y[seas:] - y[:-seas]

        v1 = np.nanvar(d1)
        vs = np.nanvar(ds)

        if vs < v1:
            return 1
        return 0

    # ---------------------------------------------------------
    # INTERNAL: mohPreEstfun CLONE
    # ---------------------------------------------------------
    def _mohPreEstfun(y_vec, seas):
        """
        Fully self-contained version of mohPreEstfun.
        """

        y = np.asarray(y_vec, dtype=float).ravel()

        # --------- 1. Randomness Test (BDS) ----------
        datatype = "Random"
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                bdsres = bdstst(y, max_dim=7, epsilon=None, distance=1)
            if np.any(bdsres[1] < 0.05):
                datatype = "Nonrandom"
        except Exception:
            datatype = "Random"

        # --------- 2. Outlier Dummy ----------
        dtry = sm.tsa.tsatools.detrend(y, order=1)
        Q1, Q3 = np.quantile(dtry, [0.25, 0.75])
        IQR = Q3 - Q1
        threshold_low = Q1 - 1.5 * IQR
        threshold_high = Q3 + 1.5 * IQR

        DumOut = np.zeros(len(y), dtype=int)
        DumOut[dtry < threshold_low] = 1
        DumOut[dtry > threshold_high] = 1

        # --------- 3. Unit Root Test (ADF) ----------
        optdiff = 0
        try:
            p = adfuller(y, regression="ct")[1]
            if p > 0.05:
                optdiff = 1
        except Exception:
            optdiff = 0

        # --------- 4. Kruskal–Wallis Seasonality Test ----------
        yKW = sm.tsa.tsatools.detrend(y, order=1)

        if seas is None:
            SeasonList = (2, 4, 6, 12, 52)
            best_p = 1.0
            final_seas = 1

            for s in SeasonList:
                KW, p = local_mohKruskalSeasTest(yKW, s)
                if p < best_p:
                    best_p = p
                    final_seas = s if p < 0.05 else 1
            finalSeasKW = final_seas
        else:
            KW, p = local_mohKruskalSeasTest(yKW, seas)
            finalSeasKW = seas if p < 0.05 else 1

        # --------- 5. Seasonal Differencing Decision (optDiff) ----------
        optDiff = local_seasonal_diff_test(y, seas)

        return datatype, DumOut, optdiff, optDiff, finalSeasKW

    # =============================================================
    # 1. Load data
    # =============================================================
    df = pd.read_excel(input_path)
    df["Date_Gregorian"] = pd.to_datetime(df["Date_Gregorian"])
    df["Vol (Kg)"] = pd.to_numeric(df["Vol (Kg)"], errors="coerce").fillna(0.0)

    required = ["Date_Gregorian", "Material Code", "Vol (Kg)"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")

    # Initialize new columns
    df["DumOut"] = np.nan
    df["datatype"] = np.nan
    df["optdiff"] = np.nan
    df["optDiff"] = np.nan
    df["finalSeasKW"] = np.nan

    # =============================================================
    # 2. Process per SKU
    # =============================================================
    grouped = df.sort_values(["Material Code", "Date_Gregorian"]).groupby("Material Code")

    for sku, g in grouped:
        idx = g.index
        y = g["Vol (Kg)"].values

        if len(y) < 3:
            df.loc[idx, ["DumOut", "datatype", "optdiff", "optDiff", "finalSeasKW"]] = \
                [0, "Random", 0, 0, 1]
            continue

        datatype, DumOut, optdiff, optDiff, finalSeasKW = _mohPreEstfun(y, seasonality)

        df.loc[idx, "DumOut"] = DumOut
        df.loc[idx, "datatype"] = datatype
        df.loc[idx, "optdiff"] = optdiff
        df.loc[idx, "optDiff"] = optDiff
        df.loc[idx, "finalSeasKW"] = finalSeasKW

    # =============================================================
    # 3. Save output
    # =============================================================
    df.to_excel(output_path, index=False)
    print(f"[INFO] Pre-estimation file written to:\n{output_path}")

    return df

## Run pre-estimation

In [ ]:
input_path = r"workspace/Clustering/Modeling_Input.xlsx"
output_path = r"workspace/PreEstimation/PreEstimation_Output.xlsx"

# Example: monthly data
seasonality = 12   # or None for auto-detection

df_pre = pre_estimate_modeling_data(
    input_path=input_path,
    seasonality=seasonality,
    output_path=output_path
)

df_pre.head()

## Extended time-series features

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf, pacf

def build_extended_features_from_preestimation(
    preest_input_path,
    output_path,
    seasonality=None,
):
    """
    Build modeling features on top of the PreEstimation output.

    IMPORTANT:
    - All features are computed ONLY from the rows that actually exist
      for each SKU (no synthetic zero months are added).

    INPUT
    -----
    preest_input_path : str
        Path to the Excel file that is the OUTPUT of your PreEstimation step.
        It must already contain, at minimum, these columns:
        - 'Date_Gregorian'
        - 'Material Code'
        - 'Vol (Kg)'
        - 'DumOut'
        - 'datatype'
        - 'optdiff'
        - 'optDiff'
        - 'finalSeasKW'
        It may also contain:
        - 'Hijri_Month_Main'
        - 'Hijri_Month_Secondary'
        - 'Sku Name'
        - 'Cluster'

    output_path : str
        Path to save the extended feature file (Excel).

    seasonality : int or None
        Global seasonality hint (e.g., 12 for monthly data with yearly seasonality).
        Used as a fallback for ACF_season if finalSeasKW <= 1.

    OUTPUT
    ------
    Returns a DataFrame with at least the following columns:

    - Date_Gregorian
    - Hijri_Month_Main
    - Hijri_Month_Secondary
    - Material Code
    - Sku Name
    - Vol (Kg)
    - Cluster
    - DumOut
    - datatype
    - optdiff
    - optDiff
    - finalSeasKW
    - ACF_season
    - Trend_slope
    - ACF_lag1
    - ACF_lag2
    - PACF_lag1
    - ADI
    - Percent_zero_demand
    """

    # ------------------------------------------------------
    # 1) Read PreEstimation output
    # ------------------------------------------------------
    df = pd.read_excel(preest_input_path)

    required_cols = [
        "Date_Gregorian",
        "Material Code",
        "Vol (Kg)",
        "DumOut",
        "datatype",
        "optdiff",
        "optDiff",
        "finalSeasKW",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"The following required columns are missing in input: {missing}")

    df["Date_Gregorian"] = pd.to_datetime(df["Date_Gregorian"])
    df.sort_values(["Material Code", "Date_Gregorian"], inplace=True)

    SKU_COL = "Material Code"
    VOL_COL = "Vol (Kg)"

    # ------------------------------------------------------
    # 2) Initialize new feature columns
    # ------------------------------------------------------
    new_cols = [
        "ACF_season",
        "Trend_slope",
        "ACF_lag1",
        "ACF_lag2",
        "PACF_lag1",
        "ADI",
        "Percent_zero_demand",
    ]
    for c in new_cols:
        df[c] = np.nan

    # ------------------------------------------------------
    # 3) Compute per-SKU features based ONLY on existing rows
    # ------------------------------------------------------
    for sku, grp in df.groupby(SKU_COL):
        grp = grp.sort_values("Date_Gregorian")
        idx = grp.index

        # Use exactly the observed volumes for this SKU
        y = grp[VOL_COL].astype(float).values
        T = len(y)
        if T == 0:
            continue

        # ---------- ADI & Percent_zero_demand ----------
        # Treat "demand months" as those with Vol > 0, based only on observed rows.
        nonzero_mask = y > 0
        n_z = int(nonzero_mask.sum())
        zeros = int((y == 0).sum())

        if n_z == 0:
            # No demand in the observed history
            adi_val = T + 1.0
        else:
            adi_val = T / float(n_z)

        percent_zero = float(zeros) / float(T) * 100.0

        # ---------- Trend_slope ----------
        if T >= 2:
            t = np.arange(T, dtype=float)
            slope, _ = np.polyfit(t, y, 1)
        else:
            slope = np.nan

        # ---------- ACF & PACF ----------
        acf_l1 = np.nan
        acf_l2 = np.nan
        acf_season = np.nan
        pacf_l1 = np.nan

        try:
            # Determine seasonal lag from finalSeasKW or global seasonality
            final_seas = grp["finalSeasKW"].iloc[0]
            if pd.isna(final_seas) or final_seas is None or final_seas <= 1:
                seas_lag = seasonality if seasonality is not None else 1
            else:
                seas_lag = int(final_seas)

            # ACF lags 1 & 2
            if T > 1:
                nlags_acf = min(max(2, seas_lag), T - 1)
                acf_vals = acf(
                    y,
                    nlags=nlags_acf,
                    fft=False,
                    missing="conservative",
                )
                if len(acf_vals) > 1:
                    acf_l1 = acf_vals[1]
                if len(acf_vals) > 2:
                    acf_l2 = acf_vals[2]

            # Seasonal ACF
            if seas_lag > 1 and T > seas_lag:
                acf_vals_full = acf(
                    y,
                    nlags=seas_lag,
                    fft=False,
                    missing="conservative",
                )
                if len(acf_vals_full) > seas_lag:
                    acf_season = acf_vals_full[seas_lag]
                else:
                    acf_season = np.nan
            else:
                # No meaningful seasonal lag → treat as 0 correlation
                acf_season = 0.0

            # PACF lag 1 (with a valid statsmodels method)
            if T > 2:
                nlags_pacf = min(2, T - 1)
                pacf_vals = pacf(y, nlags=nlags_pacf, method="yw")
                if len(pacf_vals) > 1:
                    pacf_l1 = pacf_vals[1]
            else:
                pacf_l1 = np.nan

        except Exception as e:
            print(f"[WARN] ACF/PACF computation failed for SKU {sku}: {e}")

        # Assign the same features to all rows of this SKU
        df.loc[idx, "ADI"] = adi_val
        df.loc[idx, "Percent_zero_demand"] = percent_zero
        df.loc[idx, "Trend_slope"] = slope
        df.loc[idx, "ACF_lag1"] = acf_l1
        df.loc[idx, "ACF_lag2"] = acf_l2
        df.loc[idx, "PACF_lag1"] = pacf_l1
        df.loc[idx, "ACF_season"] = acf_season

    # ------------------------------------------------------
    # 4) Reorder columns to a modeling-friendly structure
    # ------------------------------------------------------
    desired_order = [
        "Date_Gregorian",
        "Hijri_Month_Main",
        "Hijri_Month_Secondary",
        "Material Code",
        "Sku Name",
        "Vol (Kg)",
        "Cluster",
        "DumOut",
        "datatype",
        "optdiff",
        "optDiff",
        "finalSeasKW",
        "ACF_season",
        "Trend_slope",
        "ACF_lag1",
        "ACF_lag2",
        "PACF_lag1",
        "ADI",
        "Percent_zero_demand",
    ]

    main_cols = [c for c in desired_order if c in df.columns]
    other_cols = [c for c in df.columns if c not in main_cols]
    df_out = df[main_cols + other_cols]

    # ------------------------------------------------------
    # 5) Save & return
    # ------------------------------------------------------
    df_out.to_excel(output_path, index=False)
    print(f"[INFO] Extended pre-estimation features (no synthetic months) saved to: {output_path}")

    return df_out

## Run feature engineering

In [ ]:
# Paths
preest_input_path = r"workspace/PreEstimation/PreEstimation_Output.xlsx"
model_features_output_path = r"workspace/PreEstimation/PreEstimation_ModelFeatures.xlsx"

# If you want a global seasonality hint (optional; can be None)
seasonality_hint = 12  # for monthly data with yearly seasonality, or None

df_features = build_extended_features_from_preestimation(
    preest_input_path=preest_input_path,
    output_path=model_features_output_path,
    seasonality=seasonality_hint,
)

df_features.head()

## Calendar exogenous encoding

In [ ]:
import pandas as pd
import numpy as np

def build_hijri_exog_onehot(
    input_path,
    output_path,
    main_col="Hijri_Month_Main",
    secondary_col="Hijri_Month_Secondary",
    prefix_main="HijriMain",
    prefix_secondary="HijriSec",
):
    """
    Replaces Hijri_Month_Main and Hijri_Month_Secondary with one-hot encoded
    exogenous variables.

    Output columns added:
        HijriMain_1 .. HijriMain_12
        HijriSec_1  .. HijriSec_12

    Original Hijri_Month_* columns are DROPPED.
    """

    df = pd.read_excel(input_path)

    if main_col not in df.columns:
        raise ValueError(f"'{main_col}' not found in input file.")
    if secondary_col not in df.columns:
        raise ValueError(f"'{secondary_col}' not found in input file.")

    # --- Convert months safely to integers 1..12 ---
    def _clean_month(s):
        m = pd.to_numeric(s, errors="coerce")
        return m.where((m >= 1) & (m <= 12)).astype("Int64")

    df[main_col] = _clean_month(df[main_col])
    df[secondary_col] = _clean_month(df[secondary_col])

    # --- One-hot encode MAIN Hijri month ---
    main_dum = pd.get_dummies(df[main_col], prefix=prefix_main)
    for m in range(1, 13):
        col = f"{prefix_main}_{m}"
        if col not in main_dum.columns:
            main_dum[col] = 0
    main_dum = main_dum[[f"{prefix_main}_{m}" for m in range(1, 13)]].astype(int)

    # --- One-hot encode SECONDARY Hijri month ---
    sec_dum = pd.get_dummies(df[secondary_col], prefix=prefix_secondary)
    for m in range(1, 13):
        col = f"{prefix_secondary}_{m}"
        if col not in sec_dum.columns:
            sec_dum[col] = 0
    sec_dum = sec_dum[[f"{prefix_secondary}_{m}" for m in range(1, 13)]].astype(int)

    # --- Drop original Hijri columns ---
    df_out = df.drop(columns=[main_col, secondary_col])

    # --- Append exogenous variables ---
    df_out = pd.concat([df_out, main_dum, sec_dum], axis=1)

    # Save output
    df_out.to_excel(output_path, index=False)

    print(f"[INFO] Hijri exogenous variables created and saved to:")
    print(output_path)
    print("[INFO] Added 24 Hijri exogenous columns:")
    print(", ".join(main_dum.columns.tolist() + sec_dum.columns.tolist()))

    return df_out

## Run calendar encoding

In [ ]:
input_path = r"workspace/PreEstimation/PreEstimation_ModelFeatures.xlsx"
output_path = r"workspace/PreEstimation/PreEstimation_ModelFeatures_WithHijriExog.xlsx"

df_final = build_hijri_exog_onehot(
    input_path=input_path,
    output_path=output_path,
)

df_final.head()

## Multi-model forecasting and export

In [ ]:
# ==========================================================
# ALL-SKU Forecasting (NO SKU SKIPPED) + Export to Excel
# Exog columns: R:AO
# Adds model: ARIMA_Pre (per your notebook: SARIMAX with order=(1,d,1), seasonal=(1,D,1,m), trend="c")
# Output file: r"workspace/Modeling/sku_forecasts.xlsx"
# ==========================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from math import sqrt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

# ---- optional sklearn ----
SKLEARN_OK = True
try:
    from sklearn.linear_model import Ridge
    from sklearn.neural_network import MLPRegressor
    from sklearn.preprocessing import StandardScaler
except Exception:
    SKLEARN_OK = False

# ---- optional prophet ----
PROPHET_OK = True
try:
    from prophet import Prophet
except Exception:
    PROPHET_OK = False


# -------------------------
# Inputs
# -------------------------
INPUT_PATH  = r"workspace/PreEstimation/PreEstimation_ModelFeatures_WithHijriExog.xlsx"
OUTPUT_PATH = r"workspace/Modeling/sku_forecasts.xlsx"

FSTEPS      = 6
SEASONALITY = 12
BACKTEST_MONTHS = 6
TARGET_MONTH = pd.Timestamp(os.environ["FORECAST_TARGET_MONTH"]).to_period("M").start_time if os.environ.get("FORECAST_TARGET_MONTH") else None

# Small manual SARIMA grid (speed across SKUs)
GRID_P = (0, 1)
GRID_Q = (0, 1)
GRID_Ps = (0, 1)
GRID_Qs = (0, 1)
TREND_OPTIONS = ("n", "c")


# -------------------------
# Utilities
# -------------------------
def _pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of these columns found: {candidates}")
    return None

def _coerce_month_start(s):
    dt = pd.to_datetime(s, errors="coerce")
    if not hasattr(dt, "dt"):
        dt = pd.to_datetime(pd.Series(dt), errors="coerce")
    return dt.dt.to_period("M").dt.start_time

def _metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan, "sMAPE": np.nan}
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(sqrt(np.mean((y_true - y_pred) ** 2)))
    eps = 1e-9
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100.0)
    smape = float(np.mean(2.0 * np.abs(y_pred - y_true) / np.maximum(np.abs(y_true) + np.abs(y_pred), eps)) * 100.0)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "sMAPE": smape}

def _infer_exog_cols_R_AO(df):
    """
    Exog columns are R:AO in Excel.
    R is 18th col (1-indexed) => 0-index 17
    AO is 41st col (1-indexed) => 0-index 40
    slice -> [17:41]
    """
    cols = list(df.columns)
    if len(cols) >= 41:
        exog = cols[17:41]   # R..AO
    else:
        exog = [c for c in cols if ("Hijri" in c) or ("dummy" in c.lower()) or ("exog" in c.lower())]
    bad = set(["Vol (Kg)", "Material Code", "Date_Gregorian", "DumOut", "Cluster", "Sku Name"])
    return [c for c in exog if c not in bad]

def _clean_outliers_using_dumout(df_sku, y_col, dumout_col):
    s = pd.to_numeric(df_sku[y_col], errors="coerce").astype(float).copy()
    s = s.replace([np.inf, -np.inf], np.nan)

    if dumout_col is not None and dumout_col in df_sku.columns:
        is_out = pd.to_numeric(df_sku[dumout_col], errors="coerce").fillna(0).astype(int).values == 1
        base = s.copy()
        base.iloc[is_out] = np.nan
        med = float(np.nanmedian(base.values)) if np.isfinite(np.nanmedian(base.values)) else float(np.nanmedian(s.values))
        s.iloc[is_out] = med

    s = s.interpolate(limit_direction="both").fillna(0.0)
    return np.maximum(s.values, 0.0)

def _make_future_dates(last_date, fsteps):
    last = pd.Timestamp(last_date)
    return pd.date_range(last + pd.offsets.MonthBegin(1), periods=int(fsteps), freq="MS")

def _future_exog_from_history(exog_hist, m, fsteps):
    X = np.asarray(exog_hist, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    if X.shape[0] == 0:
        return np.zeros((fsteps, X.shape[1]))
    block = X[-min(m, X.shape[0]):, :] if (m and m > 1) else X[-1:, :]
    reps = int(np.ceil(fsteps / block.shape[0]))
    Xf = np.vstack([block] * reps)[:fsteps, :]
    return np.where(np.isfinite(Xf), Xf, 0.0)

def _fallback_forecast(y, h):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    if len(y) == 0:
        return np.zeros(h, dtype=float)
    last = float(y[-1])
    mean = float(np.mean(y))
    val = last if last > 0 else mean
    if not np.isfinite(val):
        val = 0.0
    return np.maximum(np.repeat(val, h), 0.0)

def _sarimax_grid_search(y, X, m, d, D, seasonal=True):
    best = None
    best_aic = np.inf
    y = np.asarray(y, dtype=float)
    X_use = None if X is None else np.asarray(X, dtype=float)

    for trend in TREND_OPTIONS:
        for p in GRID_P:
            for q in GRID_Q:
                if seasonal:
                    for P in GRID_Ps:
                        for Q in GRID_Qs:
                            order = (p, int(d), q)
                            seas  = (P, int(D), Q, int(m))
                            try:
                                res = SARIMAX(
                                    y, exog=X_use, order=order, seasonal_order=seas, trend=trend,
                                    enforce_stationarity=False, enforce_invertibility=False
                                ).fit(disp=False)
                                aic = float(res.aic)
                                if np.isfinite(aic) and aic < best_aic:
                                    best_aic = aic
                                    best = (order, seas, trend, aic)
                            except Exception:
                                pass
                else:
                    order = (p, int(d), q)
                    seas  = (0, 0, 0, 0)
                    try:
                        res = SARIMAX(
                            y, exog=X_use, order=order, seasonal_order=seas, trend=trend,
                            enforce_stationarity=False, enforce_invertibility=False
                        ).fit(disp=False)
                        aic = float(res.aic)
                        if np.isfinite(aic) and aic < best_aic:
                            best_aic = aic
                            best = (order, seas, trend, aic)
                    except Exception:
                        pass
    return best

def _fit_forecast_sarimax(y, X, h, m, d, D, use_exog, seasonal):
    best = _sarimax_grid_search(y, X if use_exog else None, m, d, D, seasonal=seasonal)
    if best is None:
        return None, {"order": None, "seasonal_order": None, "trend": None, "aic": np.nan}
    order, seas, trend, _ = best
    try:
        mod = SARIMAX(
            np.asarray(y, dtype=float),
            exog=None if not use_exog else np.asarray(X, dtype=float),
            order=order, seasonal_order=seas, trend=trend,
            enforce_stationarity=False, enforce_invertibility=False
        )
        res = mod.fit(disp=False)
        if use_exog:
            Xf = _future_exog_from_history(X, m, h)
            fc = np.asarray(res.forecast(steps=h, exog=Xf), dtype=float)
        else:
            fc = np.asarray(res.forecast(steps=h), dtype=float)
        return np.maximum(fc, 0.0), {"order": order, "seasonal_order": seas, "trend": trend, "aic": float(res.aic)}
    except Exception:
        return None, {"order": order, "seasonal_order": seas, "trend": trend, "aic": np.nan}

def _theta_forecast(y, h):
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n < 3:
        return _fallback_forecast(y, h)
    t = np.arange(n, dtype=float)
    A = np.vstack([t, np.ones(n)]).T
    b1, b0 = np.linalg.lstsq(A, y, rcond=None)[0]
    t_f = np.arange(n, n + h, dtype=float)
    lin_fc = b1 * t_f + b0
    try:
        ses = SimpleExpSmoothing(y, initialization_method="estimated").fit(optimized=True)
        ses_fc = np.asarray(ses.forecast(h), dtype=float)
    except Exception:
        ses_fc = _fallback_forecast(y, h)
    return np.maximum(0.5 * (lin_fc + ses_fc), 0.0)

def _ets_forecast(y, h, m):
    y = np.asarray(y, dtype=float)
    if len(y) < 6:
        return _fallback_forecast(y, h)
    seasonal = "add" if (m and m > 1 and len(y) >= 2*m) else None
    try:
        fit = ExponentialSmoothing(
            y, trend="add", seasonal=seasonal,
            seasonal_periods=m if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)
        return np.maximum(np.asarray(fit.forecast(h), dtype=float), 0.0)
    except Exception:
        try:
            ses = SimpleExpSmoothing(y, initialization_method="estimated").fit(optimized=True)
            return np.maximum(np.asarray(ses.forecast(h), dtype=float), 0.0)
        except Exception:
            return _fallback_forecast(y, h)

def _regression_forecast(y, X, h, m):
    if not SKLEARN_OK:
        return None, {"note": "sklearn not available"}
    y = np.asarray(y, dtype=float)
    Xex = None if X is None else np.asarray(X, dtype=float)
    if Xex is not None and Xex.ndim == 1:
        Xex = Xex.reshape(-1, 1)

    n = len(y)
    if n < 12:
        return _fallback_forecast(y, h), {"note": "short history fallback"}

    max_lag = min(12, n - 2) if (m and m > 1) else min(6, n - 2)

    rows, tgt = [], []
    for t in range(max_lag, n):
        feats = [y[t - i] for i in range(1, max_lag + 1)]
        feats.append(np.mean(y[max(0, t - m):t]) if (m and m > 1) else np.mean(y[:t]))
        if Xex is not None:
            feats.extend(Xex[t].tolist())
        rows.append(feats)
        tgt.append(y[t])

    Xsup = np.asarray(rows, dtype=float)
    ysup = np.asarray(tgt, dtype=float)
    Xsup = np.where(np.isfinite(Xsup), Xsup, 0.0)
    ysup = np.where(np.isfinite(ysup), ysup, 0.0)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(Xsup)

    model = Ridge(alpha=1.0, random_state=0)
    model.fit(Xs, ysup)

    y_hist = y.copy()
    Xf = None if Xex is None else _future_exog_from_history(Xex, m, h)

    fc = []
    for step in range(h):
        t = len(y_hist)
        feats = [y_hist[t - i] for i in range(1, max_lag + 1)]
        feats.append(np.mean(y_hist[max(0, t - m):t]) if (m and m > 1) else np.mean(y_hist))
        if Xf is not None:
            feats.extend(Xf[step].tolist())
        Xrow = np.asarray(feats, dtype=float).reshape(1, -1)
        Xrow = np.where(np.isfinite(Xrow), Xrow, 0.0)
        yhat = float(model.predict(scaler.transform(Xrow))[0])
        yhat = max(0.0, yhat)
        fc.append(yhat)
        y_hist = np.append(y_hist, yhat)

    return np.asarray(fc, dtype=float), {"note": f"ridge lag={max_lag}"}

def _ann_forecast(y, X, h, m):
    if not SKLEARN_OK:
        return None, {"note": "sklearn not available"}
    y = np.asarray(y, dtype=float)
    Xex = None if X is None else np.asarray(X, dtype=float)
    if Xex is not None and Xex.ndim == 1:
        Xex = Xex.reshape(-1, 1)

    n = len(y)
    if n < 18:
        return _fallback_forecast(y, h), {"note": "short history fallback"}

    max_lag = min(12, n - 2) if (m and m > 1) else min(6, n - 2)

    rows, tgt = [], []
    for t in range(max_lag, n):
        feats = [y[t - i] for i in range(1, max_lag + 1)]
        feats.append(np.mean(y[max(0, t - m):t]) if (m and m > 1) else np.mean(y[:t]))
        if Xex is not None:
            feats.extend(Xex[t].tolist())
        rows.append(feats)
        tgt.append(y[t])

    Xsup = np.asarray(rows, dtype=float)
    ysup = np.asarray(tgt, dtype=float)
    Xsup = np.where(np.isfinite(Xsup), Xsup, 0.0)
    ysup = np.where(np.isfinite(ysup), ysup, 0.0)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(Xsup)

    mlp = MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=350,
        random_state=0
    )
    mlp.fit(Xs, ysup)

    y_hist = y.copy()
    Xf = None if Xex is None else _future_exog_from_history(Xex, m, h)

    fc = []
    for step in range(h):
        t = len(y_hist)
        feats = [y_hist[t - i] for i in range(1, max_lag + 1)]
        feats.append(np.mean(y_hist[max(0, t - m):t]) if (m and m > 1) else np.mean(y_hist))
        if Xf is not None:
            feats.extend(Xf[step].tolist())
        Xrow = np.asarray(feats, dtype=float).reshape(1, -1)
        Xrow = np.where(np.isfinite(Xrow), Xrow, 0.0)
        yhat = float(mlp.predict(scaler.transform(Xrow))[0])
        yhat = max(0.0, yhat)
        fc.append(yhat)
        y_hist = np.append(y_hist, yhat)

    return np.asarray(fc, dtype=float), {"note": f"MLP lag={max_lag}"}

def _prophet_forecast(df_hist, date_col, y_col, exog_cols, h, m):
    if not PROPHET_OK:
        return None, {"note": "prophet not installed"}

    dff = df_hist[[date_col, y_col] + (exog_cols if exog_cols else [])].copy()
    dff = dff.rename(columns={date_col: "ds", y_col: "y"})
    dff["y"] = pd.to_numeric(dff["y"], errors="coerce").astype(float).fillna(0.0)
    dff["y"] = np.maximum(dff["y"], 0.0)

    model = Prophet(
        yearly_seasonality=True if (m and m >= 12) else False,
        weekly_seasonality=False,
        daily_seasonality=False
    )
    if exog_cols:
        for c in exog_cols:
            model.add_regressor(c)

    try:
        model.fit(dff)
    except Exception:
        return None, {"note": "prophet fit failed"}

    future = model.make_future_dataframe(periods=int(h), freq="MS")

    if exog_cols:
        Xhist = dff[exog_cols].values
        Xf = _future_exog_from_history(Xhist, m, h)
        for j, c in enumerate(exog_cols):
            future.loc[future.index[-h:], c] = Xf[:, j]

    try:
        pred = model.predict(future)
        yhat = pred["yhat"].values[-h:]
        return np.maximum(yhat, 0.0).astype(float), {"note": "prophet"}
    except Exception:
        return None, {"note": "prophet predict failed"}

# --- NEW: ARIMA_Pre (per your notebook) ---
def _arima_pre_forecast(y, h, seasonality, d, D):
    """
    Per notebook:
      order=(1,d,1)
      seasonal_order=(1,D,1,seasonality) if seasonality>1 else (0,0,0,0)
      trend="c"
      enforce_stationarity=False, enforce_invertibility=False
    Falls back to naive if short/constant or fit fails.
    """
    y = np.asarray(y, dtype=float)
    if (len(y) < 6) or (len(y) > 0 and np.all(np.isclose(y, y[0]))):
        return _fallback_forecast(y, h), {"order": (1, int(d), 1), "seasonal_order": "naive_fallback", "trend": "c"}

    order = (1, int(d), 1)
    seasonal_order = (0, 0, 0, 0)
    if int(seasonality) > 1:
        seasonal_order = (1, int(D), 1, int(seasonality))

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = SARIMAX(
                y,
                order=order,
                seasonal_order=seasonal_order,
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            res = model.fit(disp=False)
        fc = res.get_forecast(steps=int(h)).predicted_mean
        fc = np.asarray(fc, dtype=float)
        return np.maximum(fc, 0.0), {"order": order, "seasonal_order": seasonal_order, "trend": "c", "aic": float(res.aic)}
    except Exception:
        return _fallback_forecast(y, h), {"order": order, "seasonal_order": seasonal_order, "trend": "c", "note": "fit_failed_fallback"}


# -------------------------
# Load + prepare
# -------------------------
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df = pd.read_excel(INPUT_PATH)

col_date = _pick_col(df, ["Date_Gregorian", "Date", "date"])
col_mat  = _pick_col(df, ["Material Code", "Material_Code", "SKU", "sku"])
col_y    = _pick_col(df, ["Vol (Kg)", "Vol(Kg)", "Volume", "y"])
col_dum  = _pick_col(df, ["DumOut", "dumout"], required=False)
col_d    = _pick_col(df, ["optdiff", "opt_diff", "d"], required=False)
col_D    = _pick_col(df, ["optDiff", "opt_Diff", "D"], required=False)

df[col_mat] = df[col_mat].astype(str).str.strip()
df[col_date] = _coerce_month_start(df[col_date])
df = df.dropna(subset=[col_date, col_mat]).copy()
df = df.sort_values([col_mat, col_date]).reset_index(drop=True)

exog_cols = _infer_exog_cols_R_AO(df)
skus = df[col_mat].unique().tolist()

MODELS = [
    "Regression(Ridge)", "ANN(MLP)", "Theta", "ETS",
    "ARIMA_Pre",
    "ARIMA(manual)", "SARIMA(manual)", "ARIMAX(manual)", "SARIMAX(manual)", "Prophet"
]

print(f"SKUs found: {len(skus)}")
print(f"Exog columns used (R:AO): {len(exog_cols)}")
print(f"sklearn available: {SKLEARN_OK} | prophet available: {PROPHET_OK}")

all_forecast_rows = []
all_metrics_rows  = []
target_month_rows      = []

for i, sku in enumerate(skus, 1):
    g = df[df[col_mat] == sku].copy().sort_values(col_date).reset_index(drop=True)

    y = _clean_outliers_using_dumout(g, col_y, col_dum)

    X = None
    if exog_cols:
        X = g[exog_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
        X = np.where(np.isfinite(X), X, 0.0)

    d_val = int(pd.to_numeric(g[col_d].iloc[0], errors="coerce")) if col_d else 0
    D_val = int(pd.to_numeric(g[col_D].iloc[0], errors="coerce")) if col_D else 0
    d_val = 0 if not np.isfinite(d_val) else (1 if int(round(d_val)) >= 1 else 0)
    D_val = 0 if not np.isfinite(D_val) else (1 if int(round(D_val)) >= 1 else 0)

    bt = int(min(BACKTEST_MONTHS, max(1, len(y)//4))) if len(y) >= 2 else 1
    y_tr, y_te = (y[:-bt], y[-bt:]) if len(y) > bt else (y, y)
    X_tr = None if X is None else (X[:-bt] if len(X) > bt else X)

    last_hist = pd.Timestamp(g[col_date].max())
    future_dates = _make_future_dates(last_hist, FSTEPS)

    Xf = None
    if X is not None and len(exog_cols) > 0:
        Xf = _future_exog_from_history(X, SEASONALITY, FSTEPS)

    future_fc = {}

    # Regression
    fc_bt, meta = _regression_forecast(y_tr, X_tr, bt, SEASONALITY) if len(y_tr) >= 3 else (None, {"note":"too short"})
    all_metrics_rows.append({"Material Code": sku, "Model": "Regression(Ridge)", **(_metrics(y_te, fc_bt) if fc_bt is not None else {"MAE":np.nan,"RMSE":np.nan,"MAPE":np.nan,"sMAPE":np.nan}), **meta})
    fc_f, _ = _regression_forecast(y, X, FSTEPS, SEASONALITY) if len(y) >= 3 else (None, {})
    future_fc["Regression(Ridge)"] = fc_f if fc_f is not None else _fallback_forecast(y, FSTEPS)

    # ANN
    fc_bt, meta = _ann_forecast(y_tr, X_tr, bt, SEASONALITY) if len(y_tr) >= 3 else (None, {"note":"too short"})
    all_metrics_rows.append({"Material Code": sku, "Model": "ANN(MLP)", **(_metrics(y_te, fc_bt) if fc_bt is not None else {"MAE":np.nan,"RMSE":np.nan,"MAPE":np.nan,"sMAPE":np.nan}), **meta})
    fc_f, _ = _ann_forecast(y, X, FSTEPS, SEASONALITY) if len(y) >= 3 else (None, {})
    future_fc["ANN(MLP)"] = fc_f if fc_f is not None else _fallback_forecast(y, FSTEPS)

    # Theta
    fc_bt = _theta_forecast(y_tr, bt)
    all_metrics_rows.append({"Material Code": sku, "Model": "Theta", **_metrics(y_te, fc_bt)})
    future_fc["Theta"] = _theta_forecast(y, FSTEPS)

    # ETS
    fc_bt = _ets_forecast(y_tr, bt, SEASONALITY)
    all_metrics_rows.append({"Material Code": sku, "Model": "ETS", **_metrics(y_te, fc_bt)})
    future_fc["ETS"] = _ets_forecast(y, FSTEPS, SEASONALITY)

    # ARIMA_Pre (per notebook)
    fc_bt, meta = _arima_pre_forecast(y_tr, bt, SEASONALITY, d_val, D_val)
    all_metrics_rows.append({"Material Code": sku, "Model": "ARIMA_Pre", **_metrics(y_te, fc_bt), **meta})
    fc_f, _ = _arima_pre_forecast(y, FSTEPS, SEASONALITY, d_val, D_val)
    future_fc["ARIMA_Pre"] = fc_f if fc_f is not None else _fallback_forecast(y, FSTEPS)

    # ARIMA (manual grid)
    fc_bt, meta = _fit_forecast_sarimax(y_tr, None, bt, SEASONALITY, d_val, 0, use_exog=False, seasonal=False)
    all_metrics_rows.append({"Material Code": sku, "Model": "ARIMA(manual)", **(_metrics(y_te, fc_bt) if fc_bt is not None else {"MAE":np.nan,"RMSE":np.nan,"MAPE":np.nan,"sMAPE":np.nan}), **meta})
    fc_f, _ = _fit_forecast_sarimax(y, None, FSTEPS, SEASONALITY, d_val, 0, use_exog=False, seasonal=False)
    future_fc["ARIMA(manual)"] = fc_f if fc_f is not None else _fallback_forecast(y, FSTEPS)

    # SARIMA (manual grid)
    fc_bt, meta = _fit_forecast_sarimax(y_tr, None, bt, SEASONALITY, d_val, D_val, use_exog=False, seasonal=True)
    all_metrics_rows.append({"Material Code": sku, "Model": "SARIMA(manual)", **(_metrics(y_te, fc_bt) if fc_bt is not None else {"MAE":np.nan,"RMSE":np.nan,"MAPE":np.nan,"sMAPE":np.nan}), **meta})
    fc_f, _ = _fit_forecast_sarimax(y, None, FSTEPS, SEASONALITY, d_val, D_val, use_exog=False, seasonal=True)
    future_fc["SARIMA(manual)"] = fc_f if fc_f is not None else _fallback_forecast(y, FSTEPS)

    # ARIMAX (manual grid, uses exog)
    fc_bt, meta = _fit_forecast_sarimax(y_tr, X_tr, bt, SEASONALITY, d_val, 0, use_exog=True, seasonal=False) if X_tr is not None else (None, {"note":"no exog"})
    all_metrics_rows.append({"Material Code": sku, "Model": "ARIMAX(manual)", **(_metrics(y_te, fc_bt) if fc_bt is not None else {"MAE":np.nan,"RMSE":np.nan,"MAPE":np.nan,"sMAPE":np.nan}), **meta})
    fc_f, _ = _fit_forecast_sarimax(y, X, FSTEPS, SEASONALITY, d_val, 0, use_exog=True, seasonal=False) if X is not None else (None, {})
    future_fc["ARIMAX(manual)"] = fc_f if fc_f is not None else _fallback_forecast(y, FSTEPS)

    # SARIMAX (manual grid, uses exog)
    fc_bt, meta = _fit_forecast_sarimax(y_tr, X_tr, bt, SEASONALITY, d_val, D_val, use_exog=True, seasonal=True) if X_tr is not None else (None, {"note":"no exog"})
    all_metrics_rows.append({"Material Code": sku, "Model": "SARIMAX(manual)", **(_metrics(y_te, fc_bt) if fc_bt is not None else {"MAE":np.nan,"RMSE":np.nan,"MAPE":np.nan,"sMAPE":np.nan}), **meta})
    fc_f, _ = _fit_forecast_sarimax(y, X, FSTEPS, SEASONALITY, d_val, D_val, use_exog=True, seasonal=True) if X is not None else (None, {})
    future_fc["SARIMAX(manual)"] = fc_f if fc_f is not None else _fallback_forecast(y, FSTEPS)

    # Prophet
    fc_bt, meta = _prophet_forecast(g.iloc[:-bt].copy(), col_date, col_y, exog_cols, bt, SEASONALITY)
    all_metrics_rows.append({"Material Code": sku, "Model": "Prophet", **(_metrics(y_te, fc_bt) if fc_bt is not None else {"MAE":np.nan,"RMSE":np.nan,"MAPE":np.nan,"sMAPE":np.nan}), **meta})
    fc_f, _ = _prophet_forecast(g.copy(), col_date, col_y, exog_cols, FSTEPS, SEASONALITY)
    future_fc["Prophet"] = fc_f if fc_f is not None else _fallback_forecast(y, FSTEPS)

    # Forecast rows (guaranteed)
    for k in range(FSTEPS):
        row = {"Date_Gregorian": future_dates[k], "Material Code": sku}
        for mname in MODELS:
            row[mname] = float(future_fc[mname][k])
        if Xf is not None:
            for j, c in enumerate(exog_cols):
                row[c] = float(Xf[k, j])
        all_forecast_rows.append(row)

    # TargetMonth row if within horizon
    if TARGET_MONTH is not None and TARGET_MONTH in set(future_dates):
        idx = int(np.where(future_dates == TARGET_MONTH)[0][0])
        r = {"Material Code": sku, "Date_Gregorian": TARGET_MONTH}
        for mname in MODELS:
            r[mname] = float(future_fc[mname][idx])
        target_month_rows.append(r)

    if i % 10 == 0 or i == len(skus):
        print(f"Processed {i}/{len(skus)} SKUs")

# -------------------------
# Export
# -------------------------
df_forecasts = pd.DataFrame(all_forecast_rows)
df_metrics   = pd.DataFrame(all_metrics_rows)
df_target_month   = pd.DataFrame(target_month_rows)

base_cols = ["Date_Gregorian", "Material Code"] + MODELS
other_cols = [c for c in df_forecasts.columns if c not in base_cols]
df_forecasts = df_forecasts[base_cols + other_cols].sort_values(["Material Code", "Date_Gregorian"]).reset_index(drop=True)
df_metrics = df_metrics.sort_values(["Material Code", "Model"]).reset_index(drop=True)
if not df_target_month.empty:
    df_target_month = df_target_month.sort_values(["Material Code"]).reset_index(drop=True)

runinfo = pd.DataFrame([
    {"Key":"INPUT_PATH", "Value": INPUT_PATH},
    {"Key":"OUTPUT_PATH", "Value": OUTPUT_PATH},
    {"Key":"FSTEPS", "Value": FSTEPS},
    {"Key":"SEASONALITY", "Value": SEASONALITY},
    {"Key":"BACKTEST_MONTHS", "Value": BACKTEST_MONTHS},
    {"Key":"TARGET_MONTH", "Value": str(TARGET_MONTH.date()) if TARGET_MONTH is not None else "not specified"},
    {"Key":"EXOG_RANGE", "Value": "R:AO"},
    {"Key":"sklearn_available", "Value": SKLEARN_OK},
    {"Key":"prophet_available", "Value": PROPHET_OK},
    {"Key":"exog_cols_count", "Value": len(exog_cols)},
    {"Key":"Forecast_rows_expected", "Value": f"{len(skus)} * {FSTEPS} = {len(skus)*FSTEPS}"},
    {"Key":"Forecast_rows_written", "Value": int(len(df_forecasts))},
    {"Key":"Unique_SKUs_in_Forecasts", "Value": int(df_forecasts['Material Code'].nunique())},
])

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_forecasts.to_excel(writer, sheet_name="Forecasts", index=False)
    df_metrics.to_excel(writer, sheet_name="Metrics", index=False)
    df_target_month.to_excel(writer, sheet_name="TargetMonth", index=False)
    runinfo.to_excel(writer, sheet_name="RunInfo", index=False)

print("\nDONE")
print(f"Saved: {OUTPUT_PATH}")
print(f"Forecasts rows written: {len(df_forecasts)} (expected {len(skus)*FSTEPS})")
print(f"Unique SKUs in Forecasts: {df_forecasts['Material Code'].nunique()} (expected {len(skus)})")